<!-- Curated copy -->
> **Curated copy.** This notebook is taken verbatim from the BTech-thesis working archive; only
> cell *outputs* have been cleared and machine-specific absolute paths (`C:\\...`, `D:\\...`,
> `F:\\...`) have been rewritten to repository-relative `runs/...` paths. No scientific logic,
> equation, hyper-parameter or architecture has been modified. Place regenerated
> `dataset_run_*` folders under a `runs/` directory next to this notebook (or edit the paths).
> The figures this notebook originally produced are preserved in the sibling `figures/` folder.


In [ ]:
import os, json,ast, math, glob, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import glob
import pandas as pd
from sklearn.decomposition import PCA

In [ ]:
class HeatSource2DRBFPosterior:
    """
    Generate 2D heat-source fields Q(x,y) via a Gaussian-process posterior
    with an RBF (squared-exponential) kernel on a square grid.
    """

    def __init__(self, grid_size=64, length=1.0,
                 length_scale=0.20, sigma=1.0,
                 jitter=1e-6, seed=None):
        self.N = int(grid_size)
        self.L = float(length)
        self.l = float(length_scale)
        self.sigma = float(sigma)
        self.jitter = float(jitter)
        self.rng = np.random.default_rng(seed)

        x = np.linspace(0.0, self.L, self.N, dtype=np.float64)
        y = np.linspace(0.0, self.L, self.N, dtype=np.float64)
        self.X, self.Y = np.meshgrid(x, y, indexing="ij")
        self.points = np.column_stack((self.X.ravel(), self.Y.ravel()))

        # Boolean mask of boundary grid points
        self.boundary_mask = (
            np.isclose(self.X, 0.0) | np.isclose(self.X, self.L) |
            np.isclose(self.Y, 0.0) | np.isclose(self.Y, self.L)
        ).ravel()

    # RBF (Gaussian) kernel
    def rbf_kernel(self, X1, X2):
        d2 = np.sum((X1[:, None, :] - X2[None, :, :])**2, axis=-1)
        return (self.sigma**2) * np.exp(-0.5 * d2 / (self.l**2))

    # Robust Cholesky with adaptive jitter
    def safe_cholesky(self, K, max_tries=8):
        jitter = self.jitter
        I = np.eye(K.shape[0], dtype=K.dtype)
        for _ in range(max_tries):
            try:
                return np.linalg.cholesky(K + jitter * I)
            except np.linalg.LinAlgError:
                jitter *= 10.0
        raise np.linalg.LinAlgError(f"Cholesky failed; final jitter tried={jitter:g}")

    @staticmethod
    def _softplus(x, beta=6.0):
        # smooth nonnegative map; larger beta -> closer to max(0,x)
        return (1.0/beta) * np.log1p(np.exp(beta*x))

    def sample_posterior(
        self,
        Qb_func=None,
        enforce_zero_mean=True,
        *,
        force_sign=None,            # None | 'positive' | 'negative'
        pos_beta=6.0,               # sharpness for softplus
        target_mean=None,           # set desired mean after transform (e.g., 5e4)
        keep_boundary_zero=True     # keep Q=0 on the boundary after transform
    ):
        """
        Draw a sample Q from the GP posterior conditioned on boundary values Q_b.

        Parameters
        ----------
        Qb_func : callable or None
            Function returning boundary values Q_b at boundary coordinates;
            if None, boundary is conditioned to Q=0 in distribution.
        enforce_zero_mean : bool
            If True and force_sign is None, recenter to zero mean.
        force_sign : None | 'positive' | 'negative'
            Enforce all-positive or all-negative field using a smooth transform.
        pos_beta : float
            Softplus sharpness for positivity; 5–8 is typical.
        target_mean : float or None
            If given, shift the final field so mean(Q) == target_mean.
        keep_boundary_zero : bool
            If True, zero out the boundary *after* any sign/mean transforms.

        Returns
        -------
        Q : (N,N) ndarray
        """
        X_all = self.points
        X_b = X_all[self.boundary_mask]

        # Boundary values for Q
        if Qb_func is None:
            Q_b = np.zeros(len(X_b), dtype=np.float64)
        else:
            Q_b = np.asarray(Qb_func(X_b), dtype=np.float64)

        # Posterior mean and covariance
        K_xx = self.rbf_kernel(X_all, X_all).astype(np.float64)
        K_xb = self.rbf_kernel(X_all, X_b).astype(np.float64)
        K_bb = self.rbf_kernel(X_b, X_b).astype(np.float64) + self.jitter * np.eye(len(X_b))

        mu = K_xb @ np.linalg.solve(K_bb, Q_b)
        K_post = K_xx - K_xb @ np.linalg.solve(K_bb, K_xb.T)

        # Sample from posterior (Gaussian)
        Lp = self.safe_cholesky(K_post)
        z = self.rng.standard_normal(self.N * self.N)
        Q = (mu + Lp @ z).reshape(self.N, self.N)

        # Enforce sign if requested
        if force_sign is None:
            if enforce_zero_mean:
                Q -= Q.mean()
        elif force_sign == 'positive':
            Q = self._softplus(Q, beta=pos_beta)  # strictly >= 0
        elif force_sign == 'negative':
            Q = -self._softplus(Q, beta=pos_beta) # strictly <= 0
        else:
            raise ValueError("force_sign must be None, 'positive', or 'negative'")

        # Optional mean targeting *after* transform
        if target_mean is not None:
            Q += (target_mean - Q.mean())

        # Optionally keep boundary exactly zero (useful for coupling to PDE BCs)
        if keep_boundary_zero:
            Q.ravel()[self.boundary_mask] = 0.0

        return Q

    @staticmethod
    def show(Q, title="Heat source Q(x,y)", cmap="RdBu_r"):
        plt.figure(figsize=(5.0, 4.5), dpi=120)
        im = plt.imshow(Q.T, origin="lower", cmap=cmap, aspect="equal")
        plt.colorbar(im, shrink=0.9, label="Q")
        plt.title(title)
        plt.xlabel("x index")
        plt.ylabel("y index")
        plt.tight_layout()
        plt.show()


In [ ]:
#example
if __name__ == "__main__":
    gen = HeatSource2DRBFPosterior(
        grid_size=64, length=1.0, length_scale=0.18, sigma=1.0,       # length = 1.0
        jitter=1e-6, seed=7
    )

    #Q=0 on the boundary (default)
    Q0 = gen.sample_posterior(Qb_func=None, enforce_zero_mean=True)
    gen.show(Q0, title="RBF‑GP heat source (Q=0 on boundary)")

    #Heterogeneous boundary for Q (e.g., sinusoidal sides)
    def Qb(xb):
        x, y = xb[:, 0], xb[:, 1]
        return 2.0*np.sin(2*np.pi*x)*(y*(1.0-y)) + 1.5*np.sin(2*np.pi*y)*(x*(1.0-x))

    Q1 = gen.sample_posterior(Qb_func=Qb, enforce_zero_mean=True)
    gen.show(Q1, title="RBF‑GP heat source (heterogeneous boundary)")

In [ ]:
def _const_profile(val):
    """returns a function f(s) -> constant temperature array"""
    return lambda s: np.full_like(s, float(val), dtype=float)

def _gauss_profile(base, amp, mu, sigma, axis_len):
    """
    1D Gaussian along coordinate s in [0, axis_len]:
    T(s) = base + amp * exp(-0.5 * ((s - mu*axis_len)/(sigma*axis_len))**2)
    mu and sigma are given in 0..1 (relative position / width).
    """
    mu_abs = mu * axis_len
    sig_abs = max(1e-12, sigma * axis_len)
    return lambda s: base + amp * np.exp(-0.5 * ((s - mu_abs)/sig_abs)**2)

def _make_profile(side_cfg, axis_array, axis_len):
    """
    side_cfg: dict like {"on": True/False, "type": "const"/"gauss"/"custom", **params}
    axis_array: x (for top/bottom) or y (for left/right)
    axis_len: Lx (for top/bottom) or Ly (for left/right)
    returns: (on_bool, array_of_temperatures) or (False, None) if OFF
    """
    on = bool(side_cfg.get("on", False))
    if not on:
        return False, None

    typ = side_cfg.get("type", "const").lower()
    if typ == "const":
        prof = _const_profile(side_cfg.get("T", 300.0))
    elif typ == "gauss":
        prof = _gauss_profile(
            base=float(side_cfg.get("base", 330.0)),
            amp=float(side_cfg.get("amp", 20.0)),
            mu=float(side_cfg.get("mu", 0.5)),
            sigma=float(side_cfg.get("sigma", 0.2)),
            axis_len=axis_len,
        )
    elif typ == "custom":
        func = side_cfg.get("func", None)
        if not callable(func):
            raise ValueError("bc 'custom' requires a callable 'func(s_array) -> T_array'")
        prof = func
    else:
        raise ValueError(f"Unknown bc type: {typ}")

    return True, prof(axis_array)

In [ ]:
def simulate_pcm_2d_with_source(
    Q_Wm3,
    nx=128, ny=128, Lx=0.05, Ly=0.05,
    rho=800.0, cp=2000.0, k=0.2, L_lat=2e5, Tm=330.0,
    T_init=300.0, Tb=300.0,
    t_end=1200.0, cfl=0.45, save_times=(60.0, 300.0, 600.0, 1200.0),
    # ---- new: boundary configuration ----
    bc=None,
):
    """
    bc (dict) controls each side:
      bc = {
        "left":   {"on": True,  "type": "const", "T": 350.0},
        "right":  {"on": False},                                   # OFF -> adiabatic
        "bottom": {"on": True,  "type": "gauss", "base":330, "amp":20, "mu":0.5, "sigma":0.2},
        "top":    {"on": True,  "type": "custom", "func": lambda s: 335+10*np.sin(2*np.pi*s/Lx)},
      }
    If bc is None, the old behavior is kept: all four sides ON at Tb (constant).
    """

    # grid
    x = np.linspace(0.0, Lx, nx); y = np.linspace(0.0, Ly, ny)
    dx = x[1] - x[0]; dy = y[1] - y[0]
    X, Y = np.meshgrid(x, y, indexing="ij")

    # fields
    T = np.full((nx, ny), T_init, dtype=np.float64)
    f = np.zeros_like(T)
    H = cp*T + f*L_lat

    # default BCs = your previous "all Dirichlet at Tb"
    if bc is None:
        bc = {
            "left":   {"on": True,  "type": "const", "T": Tb},
            "right":  {"on": True,  "type": "const", "T": Tb},
            "bottom": {"on": True,  "type": "const", "T": Tb},
            "top":    {"on": True,  "type": "const", "T": Tb},
        }

    # precompute axis arrays for profiles
    xs = X[:, 0]        # along bottom/top edges
    ys = Y[0, :]        # along left/right edges

    # explicit stability bound for 2D FTCS
    alpha = k/(rho*cp)
    dt = cfl * min(dx*dx, dy*dy) / (4.0*alpha)

    save_times = np.array(sorted(save_times), dtype=float)
    Ts, Fs, ts, next_k = [], [], [], 0
    t = 0.0

    def apply_BCs(Tarr, Harr):
        """
        Apply per-side boundary modes:
          - ON  -> Dirichlet with given profile
          - OFF -> adiabatic (copy interior)
        Re-sync H on Dirichlet sides.
        """
        # LEFT
        on, arr = _make_profile(bc.get("left", {}), ys, Ly)
        if on:
            Tarr[0, :] = arr
            Harr[0, :] = cp*Tarr[0, :] + f[0, :]*L_lat
        else:
            Tarr[0, :] = Tarr[1, :]  # adiabatic

        # RIGHT
        on, arr = _make_profile(bc.get("right", {}), ys, Ly)
        if on:
            Tarr[-1, :] = arr
            Harr[-1, :] = cp*Tarr[-1, :] + f[-1, :]*L_lat
        else:
            Tarr[-1, :] = Tarr[-2, :]

        # BOTTOM
        on, arr = _make_profile(bc.get("bottom", {}), xs, Lx)
        if on:
            Tarr[:, 0] = arr
            Harr[:, 0] = cp*Tarr[:, 0] + f[:, 0]*L_lat
        else:
            Tarr[:, 0] = Tarr[:, 1]

        # TOP
        on, arr = _make_profile(bc.get("top", {}), xs, Lx)
        if on:
            Tarr[:, -1] = arr
            Harr[:, -1] = cp*Tarr[:, -1] + f[:, -1]*L_lat
        else:
            Tarr[:, -1] = Tarr[:, -2]

    # initial boundary imposition
    apply_BCs(T, H)

    while t < t_end + 1e-12:
        # enforce BCs before building Laplacian (safety)
        apply_BCs(T, H)

        # five-point Laplacian (interior only)
        Lap = np.zeros_like(T)
        Lap[1:-1,1:-1] = (
            (T[2:,1:-1] - 2*T[1:-1,1:-1] + T[:-2,1:-1]) / dx**2 +
            (T[1:-1,2:] - 2*T[1:-1,1:-1] + T[1:-1,:-2]) / dy**2
        )

        # enthalpy update (interior only) with heat source
        H[1:-1,1:-1] += dt * ((k/rho) * Lap[1:-1,1:-1] + Q_Wm3[1:-1,1:-1] / rho)

        # phase update
        f_new = (H - cp*Tm) / L_lat
        mush  = (f_new > 0.0) & (f_new < 1.0)
        solid = (f_new <= 0.0)
        liq   = (f_new >= 1.0)

        T[mush]  = Tm
        f[mush]  = f_new[mush]
        T[solid] = H[solid]/cp
        f[solid] = 0.0
        T[liq]   = (H[liq]-L_lat)/cp
        f[liq]   = 1.0

        # re-apply BCs and resync H on Dirichlet sides
        apply_BCs(T, H)

        # save snapshots
        if next_k < len(save_times) and t >= save_times[next_k] - 0.5*dt:
            Ts.append(T.copy()); Fs.append(f.copy()); ts.append(t); next_k += 1

        t += dt

    return np.array(Ts), np.array(Fs), np.array(ts), X, Y, {"dt": dt}

In [ ]:
def plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm, Q, qlabel="Q [W/m^3]",colormap="RdBu_r"):
    """
    Show heat source Q plus temperature and liquid fraction snapshots.
    Rows: 0=Q (same in all columns), 1=T, 2=f
    Columns: different saved times.
    """
    n = len(ts)
    fig, axes = plt.subplots(3, n, figsize=(4.4*n, 10), dpi=120, constrained_layout=True)

    x0, x1 = float(X.min()), float(X.max())
    y0, y1 = float(Y.min()), float(Y.max())

    #Q row 0
    for k in range(n):
        imQ = axes[0, k].imshow(Q.T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap=colormap,
                                aspect="equal")
        axes[0, k].set_title("Heat source Q")
        axes[0, k].set_xlabel("x [m]"); axes[0, k].set_ylabel("y [m]")
        if k == 0:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85, label=qlabel)
        else:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85)

    # temperature with T=Tm contour row 1
    for k in range(n):
        imT = axes[1, k].imshow(Ts[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="inferno", aspect="equal")
        axes[1, k].contour(X, Y, Ts[k], levels=[Tm], colors='cyan', linewidths=1.0)
        axes[1, k].set_title(f"T at t={ts[k]:.1f} s")
        axes[1, k].set_xlabel("x [m]"); axes[1, k].set_ylabel("y [m]")
        fig.colorbar(imT, ax=axes[1, k], shrink=0.85, label="T [K]")

    #Liquid fraction f row2
    for k in range(n):
        imF = axes[2, k].imshow(Fs[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="viridis", vmin=0.0, vmax=1.0, aspect="equal")
        axes[2, k].set_title(f"Liquid fraction at t={ts[k]:.1f} s")
        axes[2, k].set_xlabel("x [m]"); axes[2, k].set_ylabel("y [m]")
        fig.colorbar(imF, ax=axes[2, k], shrink=0.85, label="f [-]")

    plt.show()

In [ ]:
T_m=330
t_end=4000.0
T_bound=350.0
T_initial=300.0

N=120
nx, ny = N,N
Lx, Ly = 0.05, 0.05
x = np.linspace(0.0, Lx, nx)
y = np.linspace(0.0, Ly, ny)

bc = {
    "left":   {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.5, "sigma":0.2},  # initial values - 20, 0.5, 0.2
    "right":  {"on": True, "type": "const", "T": 330.0},
    "bottom": {"on": True, "type": "const", "T": 340.0},
    "top":    {"on": True, "type": "const", "T": 350.0},
}
onL, TL = _make_profile(bc.get("left",   {}), y, Ly)
onR, TR = _make_profile(bc.get("right",  {}), y, Ly)
onB, TB = _make_profile(bc.get("bottom", {}), x, Lx)
onT, TT = _make_profile(bc.get("top",    {}), x, Lx)

# plot
fig, ax = plt.subplots(2, 2, figsize=(8,6), constrained_layout=True)
if onL: ax[0,0].plot(y, TL, lw=2); ax[0,0].set_title("Left (x=0)");   ax[0,0].set_xlabel("y [m]"); ax[0,0].set_ylabel("T [K]")
else:   ax[0,0].set_title("Left OFF (adiabatic)")

if onR: ax[0,1].plot(y, TR, lw=2); ax[0,1].set_title("Right (x=Lx)"); ax[0,1].set_xlabel("y [m]"); ax[0,1].set_ylabel("T [K]")
else:   ax[0,1].set_title("Right OFF (adiabatic)")

if onB: ax[1,0].plot(x, TB, lw=2); ax[1,0].set_title("Bottom (y=0)"); ax[1,0].set_xlabel("x [m]"); ax[1,0].set_ylabel("T [K]")
else:   ax[1,0].set_title("Bottom OFF (adiabatic)")

if onT: ax[1,1].plot(x, TT, lw=2); ax[1,1].set_title("Top (y=Ly)");   ax[1,1].set_xlabel("x [m]"); ax[1,1].set_ylabel("T [K]")
else:   ax[1,1].set_title("Top OFF (adiabatic)")

for a in ax.ravel(): a.grid(alpha=0.25)
plt.show()

In [ ]:
# RUN TO CHECK ONLY CREATED FIELD SAMPLES

N=40
genQ = HeatSource2DRBFPosterior(grid_size=N, length=1.0, length_scale=0.18, sigma=1.0, jitter=1e-6, seed=None)
Q_dimless = genQ.sample_posterior(
    enforce_zero_mean=False,
    force_sign='positive',   # ensures nonnegative field
    target_mean=1,         # dimensionless mean
    keep_boundary_zero=False,
)
#scaling to physical W/m^3 magnitude for source strength
q_scale = 5e5   # W/m^3
Q = q_scale * Q_dimless

Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(Q_Wm3=Q,nx=N, ny=N, Lx=0.05, Ly=0.05,
                                                    rho=800.0, cp=2000.0, k=0.2, L_lat=2e5,
                                                    t_end=t_end, cfl=0.45, bc=bc, T_init=T_initial, Tm=T_m, save_times=(10.0,60.0, 300.0, 600.0, 1200.0, 2000.0,2500.0))
plot_fields_with_Q(Ts, Fs, ts, X, Y, T_m, Q, qlabel="Q",colormap="RdBu_r")

In [ ]:
'''# --- knobs you already use ---
T_m      = 330.0
t_end    = 4000.0
T_bound  = 350.0
T_initial= 300.0

N  = 120
nx = ny = N
Lx = Ly = 0.05
x  = np.linspace(0.0, Lx, nx)
y  = np.linspace(0.0, Ly, ny)

rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
cfl = 0.45
save_times = (10.0, 60.0, 300.0, 600.0, 1200.0, 2000.0, 2500.0)

q_scale = 5e4   # W/m^3
Q_length_scale = 0.18
Q_sigma        = 1.0

# ------------ helper: plot boundary profiles (reuses your _make_profile) ------------
def plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=None, savepath=None):
    onL, TL = _make_profile(bc.get("left",   {}), y, Ly)
    onR, TR = _make_profile(bc.get("right",  {}), y, Ly)
    onB, TB = _make_profile(bc.get("bottom", {}), x, Lx)
    onT, TT = _make_profile(bc.get("top",    {}), x, Lx)

    fig, ax = plt.subplots(2, 2, figsize=(8,6), constrained_layout=True)
    if figtitle: fig.suptitle(figtitle)

    if onL: ax[0,0].plot(y, TL, lw=2); ax[0,0].set_title("Left (x=0)")
    else:   ax[0,0].set_title("Left OFF (adiabatic)")
    ax[0,0].set_xlabel("y [m]"); ax[0,0].set_ylabel("T [K]")

    if onR: ax[0,1].plot(y, TR, lw=2); ax[0,1].set_title("Right (x=Lx)")
    else:   ax[0,1].set_title("Right OFF (adiabatic)")
    ax[0,1].set_xlabel("y [m]"); ax[0,1].set_ylabel("T [K]")

    if onB: ax[1,0].plot(x, TB, lw=2); ax[1,0].set_title("Bottom (y=0)")
    else:   ax[1,0].set_title("Bottom OFF (adiabatic)")
    ax[1,0].set_xlabel("x [m]"); ax[1,0].set_ylabel("T [K]")

    if onT: ax[1,1].plot(x, TT, lw=2); ax[1,1].set_title("Top (y=Ly)")
    else:   ax[1,1].set_title("Top OFF (adiabatic)")
    ax[1,1].set_xlabel("x [m]"); ax[1,1].set_ylabel("T [K]")

    for a in ax.ravel(): a.grid(alpha=0.25)
    if savepath:
        plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.show()

# ------------ boundary-case generator (10 combos) ------------
def make_bc_cases(T_bound, Lx, Ly):
    cases = []

    # 1) All sides hot (const), left as Gaussian
    cases.append({
        "left":   {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.5, "sigma":0.20},
        "right":  {"on": True, "type": "const", "T": T_bound},
        "bottom": {"on": True, "type": "const", "T": T_bound},
        "top":    {"on": True, "type": "const", "T": T_bound},
    })

    # 2–4) One-side hot each time
    cases.append({"left":  {"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "bottom":{"on": False}, "top":{"on": False}})
    cases.append({"right": {"on": True, "type": "const", "T": T_bound},
                  "left":  {"on": False}, "bottom":{"on": False}, "top":{"on": False}})
    cases.append({"bottom":{"on": True, "type": "const", "T": T_bound},
                  "left":  {"on": False}, "right":{"on": False}, "top":{"on": False}})

    # 5) Left Gaussian, top const, others off
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.5, "sigma":0.20},
                  "top":   {"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "bottom":{"on": False}})

    # 6–8) Vary left Gaussian position/width
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.3, "sigma":0.15},
                  "right": {"on": True, "type": "const", "T": T_bound},
                  "bottom":{"on": False}, "top":{"on": False}})
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.7, "sigma":0.25},
                  "bottom":{"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "top":{"on": False}})
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":25, "mu":0.5, "sigma":0.10},
                  "top":   {"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "bottom":{"on": False}})

    # 9) Opposite sides hot (right & bottom), left Gaussian off
    cases.append({"left":  {"on": False},
                  "right": {"on": True, "type": "const", "T": T_bound},
                  "bottom":{"on": True, "type": "const", "T": T_bound},
                  "top":   {"on": False}})

    # 10) Checker: left Gaussian + right const + top const
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":15, "mu":0.6, "sigma":0.18},
                  "right": {"on": True, "type": "const", "T": T_bound},
                  "top":   {"on": True, "type": "const", "T": T_bound},
                  "bottom":{"on": False}})
    return cases

bc_cases = make_bc_cases(T_bound, Lx, Ly)

# ------------ batch loop ------------
for i, bc in enumerate(bc_cases, 1):
    title = f"BC Case {i}"
    # 1) Boundary profiles
    plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=title,
                           savepath=f"bc_profiles_case{i}.png")

    # 2) Positive heat source (static in time)
    genQ = HeatSource2DRBFPosterior(grid_size=N, length=1.0,
                                    length_scale=Q_length_scale, sigma=Q_sigma,
                                    jitter=1e-6, seed=1234+i)  # seed per case for variety

    Q_dimless = genQ.sample_posterior(
        enforce_zero_mean=False,
        force_sign='positive',     # strictly >=0
        target_mean=1.0,           # dimensionless mean ≈ 1
        keep_boundary_zero=False   # allow interior & edges to carry source
    )
    Q = q_scale * Q_dimless

    # 3) Simulate
    Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
        Q_Wm3=Q, nx=N, ny=N, Lx=Lx, Ly=Ly,
        rho=rho, cp=cp, k=k, L_lat=L_lat,
        t_end=t_end, cfl=cfl, bc=bc, T_init=T_initial, Tm=T_m,
        save_times=save_times
    )

    # 4) Plot maps (use your function signature; change colormap as you like)
    plot_fields_with_Q(Ts, Fs, ts, X, Y, T_m, Q, qlabel="Q [W/m^3]", colormap="Reds")
    plt.savefig(f"maps_case{i}.png", dpi=200, bbox_inches="tight")
    plt.close('all')'''


In [ ]:
#T_m       = 330.0
#t_end     = 4000.0
#T_bound   = 350.0
#T_initial = 300.0
#
#N  = 120
#nx = ny = N
#Lx = Ly = 0.05
#x  = np.linspace(0.0, Lx, nx)
#y  = np.linspace(0.0, Ly, ny)
#
#rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
#cfl = 0.45
#save_times = (10.0, 60.0, 300.0, 600.0, 1200.0, 2000.0, 2500.0)
#
## heat-source GP params
#q_scale        = 5e4       # W/m^3
#Q_length_scale = 0.18
#Q_sigma        = 1.0
#
#
#def plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=None, savepath=None):
#    onL, TL = _make_profile(bc.get("left",   {}), y, Ly)
#    onR, TR = _make_profile(bc.get("right",  {}), y, Ly)
#    onB, TB = _make_profile(bc.get("bottom", {}), x, Lx)
#    onT, TT = _make_profile(bc.get("top",    {}), x, Lx)
#
#    fig, ax = plt.subplots(2, 2, figsize=(8,6), constrained_layout=True)
#    if figtitle: fig.suptitle(figtitle)
#
#    ax[0,0].plot(y, TL, lw=2); ax[0,0].set_title("Left (x=0)")
#    ax[0,1].plot(y, TR, lw=2); ax[0,1].set_title("Right (x=Lx)")
#    ax[1,0].plot(x, TB, lw=2); ax[1,0].set_title("Bottom (y=0)")
#    ax[1,1].plot(x, TT, lw=2); ax[1,1].set_title("Top (y=Ly)")
#
#    for a in ax.ravel():
#        a.set_xlabel(("y [m]" if a in (ax[0,0], ax[0,1]) else "x [m]"))
#        a.set_ylabel("T [K]")
#        a.grid(alpha=0.25)
#
#    if savepath:
#        plt.savefig(savepath, dpi=200, bbox_inches="tight")
#    plt.show()
#
## make 15 BC cases (ALL sides ON)
#def make_bc_cases_all_on(T_bound, n_cases=15):
#    # cyclic parameter grids (repeat as needed)
#    mus    = np.linspace(0.25, 0.75, 5)          # Gaussian centers (relative)
#    sigmas = np.linspace(0.12, 0.28, 3)          # Gaussian widths (relative)
#    amps   = [12.0, 18.0, 24.0]                  # amplitude (K)
#
#    cases = []
#    for i in range(n_cases):
#        # choose parameters cyclically for variety
#        muL,  muR  = mus[i % len(mus)], mus[(i+2) % len(mus)]
#        muB,  muT  = mus[(i+1) % len(mus)], mus[(i+3) % len(mus)]
#        sL,   sR   = sigmas[i % len(sigmas)], sigmas[(i+1) % len(sigmas)]
#        sB,   sT   = sigmas[(i+2) % len(sigmas)], sigmas[(i+0) % len(sigmas)]
#        aL,   aR   = amps[i % len(amps)], amps[(i+1) % len(amps)]
#        aB,   aT   = amps[(i+2) % len(amps)], amps[(i+0) % len(amps)]
#
#        # alternate sides between Gaussian and constant—but ON for all
#        if i % 3 == 0:
#            left   = {"on": True, "type": "gauss", "base": T_bound, "amp": aL, "mu": muL, "sigma": sL}
#            right  = {"on": True, "type": "const", "T": T_bound}
#            bottom = {"on": True, "type": "gauss", "base": T_bound, "amp": aB, "mu": muB, "sigma": sB}
#            top    = {"on": True, "type": "const", "T": T_bound}
#        elif i % 3 == 1:
#            left   = {"on": True, "type": "const", "T": T_bound}
#            right  = {"on": True, "type": "gauss", "base": T_bound, "amp": aR, "mu": muR, "sigma": sR}
#            bottom = {"on": True, "type": "const", "T": T_bound}
#            top    = {"on": True, "type": "gauss", "base": T_bound, "amp": aT, "mu": muT, "sigma": sT}
#        else:
#            left   = {"on": True, "type": "gauss", "base": T_bound, "amp": aL, "mu": muL, "sigma": sL}
#            right  = {"on": True, "type": "gauss", "base": T_bound, "amp": aR, "mu": muR, "sigma": sR}
#            bottom = {"on": True, "type": "const", "T": T_bound}
#            top    = {"on": True, "type": "const", "T": T_bound}
#
#        cases.append({"left": left, "right": right, "bottom": bottom, "top": top})
#    return cases
#
#bc_cases = make_bc_cases_all_on(T_bound, n_cases=15)
#
## ---------- batch loop ----------
#for i, bc in enumerate(bc_cases, 1):
#    title = f"BC Case {i:02d} (all sides ON)"
#    plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=title,
#                           savepath=f"bc_profiles_case{i:02d}.png")
#
#    # positive heat-source (static in time), new seed per case
#    genQ = HeatSource2DRBFPosterior(
#        grid_size=N, length=1.0,
#        length_scale=Q_length_scale, sigma=Q_sigma,
#        jitter=1e-6, seed=1000+i
#    )
#    Q_dimless = genQ.sample_posterior(
#        enforce_zero_mean=False,
#        force_sign='positive',    # strictly >= 0
#        target_mean=1.0,
#        keep_boundary_zero=False  # source not forced to zero on edges
#    )
#    Q = q_scale * Q_dimless
#
#    Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
#        Q_Wm3=Q, nx=N, ny=N, Lx=Lx, Ly=Ly,
#        rho=rho, cp=cp, k=k, L_lat=L_lat,
#        t_end=t_end, cfl=cfl, bc=bc, T_init=T_initial, Tm=T_m,
#        save_times=save_times
#    )
#
#    plot_fields_with_Q(Ts, Fs, ts, X, Y, T_m, Q, qlabel="Q [W/m^3]", colormap="Reds")
#    plt.savefig(f"maps_case{i:02d}.png", dpi=200, bbox_inches="tight")
#    plt.close('all')

In [ ]:
'''def simulate_pcm_2d_with_source(
    Q_Wm3, nx=128, ny=128, Lx=0.05, Ly=0.05,
    rho=800.0, cp=2000.0, k=0.2, L_lat=2e5, Tm=330.0,
    T_init=300.0, Tb=300.0,
    t_end=1200.0, cfl=0.45, save_times=(60.0, 300.0, 600.0, 1200.0)
):
    # grid
    x = np.linspace(0.0, Lx, nx); y = np.linspace(0.0, Ly, ny)
    dx = x[1] - x[0]; dy = y[1] - y[0]
    X, Y = np.meshgrid(x, y, indexing="ij")

    # fields
    T = np.full((nx, ny), T_init, dtype=np.float64)
    f = np.zeros_like(T)
    H = cp*T + f*L_lat

    # to impose Dirichlet temperature on all walls once before the loop
    T[0,:]=T[-1,:]=Tb; T[:,0]=T[:,-1]=Tb
    H[0,:]=cp*Tb + f[0,:]*L_lat
    H[-1,:]=cp*Tb + f[-1,:]*L_lat
    H[:,0]=cp*Tb + f[:,0]*L_lat
    H[:,-1]=cp*Tb + f[:,-1]*L_lat

    # explicit stability bound for 2D FTCS:  that 1/4(dx^2+dy^2)/alpha condition taught in cfd
    alpha = k/(rho*cp)
    dt = cfl * min(dx*dx, dy*dy) / (4.0*alpha)

    save_times = np.array(sorted(save_times), dtype=float)
    Ts, Fs, ts, next_k = [], [], [], 0
    t = 0.0

    while t < t_end + 1e-12:
        # to always enforce wall temperature before forming Laplacian as a safety checker
        T[0,:]=T[-1,:]=Tb; T[:,0]=T[:,-1]=Tb

        # five-point Laplacian-interior only formation
        Lap = np.zeros_like(T)
        Lap[1:-1,1:-1] = (
            (T[2:,1:-1] - 2*T[1:-1,1:-1] + T[:-2,1:-1]) / dx**2 +
            (T[1:-1,2:] - 2*T[1:-1,1:-1] + T[1:-1,:-2]) / dy**2
        )

        #enthalpy update: interior only, with heat source
        H[1:-1,1:-1] += dt * ((k/rho) * Lap[1:-1,1:-1] + Q_Wm3[1:-1,1:-1] / rho)

        #returns boolean arrays here in 2d space as masks
        f_new = (H - cp*Tm) / L_lat
        mush  = (f_new > 0.0) & (f_new < 1.0)
        solid = (f_new <= 0.0)
        liq   = (f_new >= 1.0)

        T[mush]  = Tm;
        f[mush]  = f_new[mush]
        T[solid] = H[solid]/cp
        f[solid] = 0.0
        T[liq]   = (H[liq]-L_lat)/cp
        f[liq] = 1.0

        #reimpose Dirichlet temperature on walls and resync H values
        T[0,:]=T[-1,:]=Tb; T[:,0]=T[:,-1]=Tb
        H[0,:]  = cp*Tb + f[0,:]*L_lat
        H[-1,:] = cp*Tb + f[-1,:]*L_lat
        H[:,0]  = cp*Tb + f[:,0]*L_lat
        H[:,-1] = cp*Tb + f[:,-1]*L_lat

        #save snapshots
        if next_k < len(save_times) and t >= save_times[next_k] - 0.5*dt:
            Ts.append(T.copy()); Fs.append(f.copy()); ts.append(t); next_k += 1

        t += dt

    return np.array(Ts), np.array(Fs), np.array(ts), X, Y, {"dt": dt}

def plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm, Q, qlabel="Q [W/m^3]"):
    """
    Show heat source Q plus temperature and liquid fraction snapshots.
    Rows: 0=Q (same in all columns), 1=T, 2=f
    Columns: different saved times.
    """
    n = len(ts)
    fig, axes = plt.subplots(3, n, figsize=(4.4*n, 10), dpi=120, constrained_layout=True)

    x0, x1 = float(X.min()), float(X.max())
    y0, y1 = float(Y.min()), float(Y.max())

    #Q row 0
    for k in range(n):
        imQ = axes[0, k].imshow(Q.T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="RdBu_r", aspect="equal")
        axes[0, k].set_title("Heat source Q")
        axes[0, k].set_xlabel("x [m]"); axes[0, k].set_ylabel("y [m]")
        if k == 0:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85, label=qlabel)
        else:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85)

    # temperature with T=Tm contour row 1
    for k in range(n):
        imT = axes[1, k].imshow(Ts[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="inferno", aspect="equal")
        axes[1, k].contour(X, Y, Ts[k], levels=[Tm], colors='cyan', linewidths=1.0)
        axes[1, k].set_title(f"T at t={ts[k]:.1f} s")
        axes[1, k].set_xlabel("x [m]"); axes[1, k].set_ylabel("y [m]")
        fig.colorbar(imT, ax=axes[1, k], shrink=0.85, label="T [K]")

    #Liquid fraction f row2
    for k in range(n):
        imF = axes[2, k].imshow(Fs[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="viridis", vmin=0.0, vmax=1.0, aspect="equal")
        axes[2, k].set_title(f"Liquid fraction at t={ts[k]:.1f} s")
        axes[2, k].set_xlabel("x [m]"); axes[2, k].set_ylabel("y [m]")
        fig.colorbar(imF, ax=axes[2, k], shrink=0.85, label="f [-]")

    plt.show()'''


In [ ]:
'''if __name__ == "__main__":
    N = 40
    genQ = HeatSource2DRBFPosterior(grid_size=N, length=1.0, length_scale=0.18, sigma=1.0, jitter=1e-6, seed=None)
    Q_dimless = genQ.sample_posterior(enforce_zero_mean=False, force_sign="positive",target_mean=5e4,keep_boundary_zero=True)

    #scaling to physical W/m^3 magnitude for source strength
    #tune the parameters as needed here
    q_scale = 5e4    # W/m^3
    Q = q_scale * Q_dimless
    T_m=330
    t_end=4000.0
    T_bound=350.0
    T_initial=300.0

    Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
        Q_Wm3=Q, nx=N, ny=N, Lx=0.05, Ly=0.05,
        rho=800.0, cp=2000.0, k=0.2, L_lat=3e5, Tm=T_m,
        T_init=T_initial,Tb=T_bound, t_end=t_end, cfl=0.45, save_times=(10.0,60.0, 300.0, 600.0, 1200.0, 2000.0,3000.0,3900.0)
    )
    print(f"dt = {info['dt']:.4e} s")
    plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm=T_m,Q=Q, qlabel="Q [W/m³]")'''

### DATASET GENERATOR


In [ ]:
N        = 40          # grid points in x,y (Q, T, f are NxN)
Lx, Ly   = 0.05, 0.05
rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
T_m      = 330.0
T_init   = 300.0
T_bound  = 330.0
t_end    = 4000.0
save_times = (100.0, 250.0, 400.0, 600.0, 1000.0, 1200.0, 1500.0, 1800.0, 2100.0)
cfl      = 0.45

# Heat-source GP
q_scale        = 1e5     # W/m^3
Q_length_scale = 0.18
Q_sigma        = 1.0

# Boundary Conditions
mu_max=0.8
sigma_max=0.7      
amp_max=80

# Dataset sizes
NUM_CASES      = 200.0      # total function-realizations / cases
TRAIN_FRAC     = 0.8
VAL_FRAC       = 0.1      # test is the remainder

# DeepONet sampling
sensor_mode    = "full"   # "full" (flatten full NxN) or "downsample"
S_down         = 40       # used if sensor_mode="downsample" (S_down x S_down grid)
n_time_bc      = len(save_times)
S_down_BC      = N

points_per_case_per_time = N**2  # # of (x,y) points sampled per time snapshot

# Boundary configuration control
only_lr_vary   = True     # True: top & bottom constant, left/right varied; False: allow all configurable
All_side_const_temp_boundary = False    # if all sides constant boundary is wanted at T_bound for dataset testing

# MODEL PARAMETERS

BATCH_POINTS = 32768//2
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4# first case 0.0
VAL_SAMPLES = 4
VAL_BATCH_POINTS = 32768//2
STEPS_PER_EPOCH = 64

#each path weight
alpha_add=1.0
alpha_prod=0.2

In [ ]:
# Output directory
RUN_DIR = Path(f"dataset_run_{time.strftime('%Y%m%d-%H%M%S')}")
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def make_bc_case(i, T_bound=350.0, only_lr=True, all_const=False):
    """
    i: case index for deterministic variety
    only_lr=True -> top/bottom constant; left/right varied (gaussian/const)
    """
    rng = np.random.default_rng(10_000 + i)

    # const sides (top, bottom)
    bottom = {"on": True, "type": "const", "T": T_bound}
    top    = {"on": True, "type": "const", "T": T_bound}

    # varied sides: either const or gaussian around T_bound
    def rand_gauss():
        mu    = rng.uniform(0.25, mu_max)
        sigma = rng.uniform(0.12, sigma_max)
        amp   = rng.uniform(12.0, amp_max)
        return {"on": True, "type": "gauss", "base": T_bound, "amp": float(amp),
                "mu": float(mu), "sigma": float(sigma)}

    def rand_const():
        # small jitter around T_bound if you like; here keep exactly T_bound
        return {"on": True, "type": "const", "T": T_bound}

    left  = rand_gauss() if (i % 2 == 0) else rand_const()
    right = rand_gauss() if (i % 3 == 0) else rand_const()

    if not only_lr:
        # Optionally make top/bottom also vary (still ON)
        if (i % 5) == 0:
            top = rand_gauss()
        if (i % 7) == 0:
            bottom = rand_gauss()
    
    if all_const:
        bottom = {"on": True, "type": "const", "T": T_bound}
        top    = {"on": True, "type": "const", "T": T_bound}
        right = {"on": True, "type": "const", "T": T_bound}
        left = {"on": True, "type": "const", "T": T_bound}

    return {"left": left, "right": right, "bottom": bottom, "top": top}

# --- helper: branch sensorizing ---
def sensorize_field(arr2d, mode="full", S=40):
    """
    Converts a 2D field (NxN) to a branch vector.
    mode="full": flatten (N*N,)
    mode="downsample": sample on an SxS regular grid
    """
    if mode == "full":
        return arr2d.reshape(-1).astype(np.float32)
    elif mode == "downsample":
        N = arr2d.shape[0]
        xs = np.linspace(0, N-1, S).round().astype(int)
        ys = np.linspace(0, N-1, S).round().astype(int)
        sub = arr2d[np.ix_(xs, ys)]
        return sub.reshape(-1).astype(np.float32)
    else:
        raise ValueError("sensor_mode must be 'full' or 'downsample'")

# --- helper: sample trunk points for a case ---
#def sample_trunk_points(N, Lx, Ly, times, per_time, rng):
#    """
#    Returns: coords [M,3] (x,y,t), idxs [M,2] (i,j indices on grid), time_ids [M]
#    where M = per_time * len(times)
#    """
#    xs = np.linspace(0.0, Lx, N)
#    ys = np.linspace(0.0, Ly, N)
#
#    coords_list, ij_list, tid_list = [], [], []
#    for ti, t in enumerate(times):
#        i_idx = rng.integers(0, N, size=per_time)
#        j_idx = rng.integers(0, N, size=per_time)
#        x_samp = xs[i_idx]
#        y_samp = ys[j_idx]
#        t_samp = np.full_like(x_samp, float(t), dtype=np.float64)
#
#        coords_list.append(np.stack([x_samp, y_samp, t_samp], axis=1))
#        ij_list.append(np.stack([i_idx, j_idx], axis=1))
#        tid_list.append(np.full(per_time, ti, dtype=np.int32))
#
#    coords = np.concatenate(coords_list, axis=0)             # [M,3]
#    ij     = np.concatenate(ij_list, axis=0).astype(np.int32)# [M,2]
#    tids   = np.concatenate(tid_list, axis=0).astype(np.int32)
#    return coords, ij, tids
#

def sample_trunk_points_split(N, Lx, Ly, times, per_time, rng):
    """
    Returns:
      xy     [M,2]  -> (x,y)
      t      [M,1]  -> (t,)
      ij     [M,2]  -> (row=j, col=i) for array indexing [j,i]
      tids   [M]    -> time index into `times`
    where M = per_time * len(times)
    """
    if rng is None:
        rng = np.random.default_rng()

    times = np.asarray(times, dtype=float)
    M = per_time * len(times)

    # integer grid indices
    js = rng.integers(0, N, size=M)    # row (y)
    is_ = rng.integers(0, N, size=M)   # col (x)

    # physical coordinates
    dx = Lx / (N - 1 if N > 1 else 1)
    dy = Ly / (N - 1 if N > 1 else 1)
    xs = is_ * dx
    ys = js  * dy

    # times per block
    tids = np.repeat(np.arange(len(times), dtype=np.int32), per_time)
    t    = times[tids].reshape(-1, 1).astype(np.float32)

    xy = np.column_stack([xs, ys]).astype(np.float32)
    ij = np.column_stack([js, is_]).astype(np.int32)
    return xy, t, ij, tids


In [ ]:
import os, json
from pathlib import Path

def plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=None, savepath=None):
    """
    Line plots of boundary temperature profiles:
      Top (y=Ly)    : vs x
      Bottom (y=0)  : vs x
      Left (x=0)    : vs y
      Right (x=Lx)  : vs y
    Uses your gauss/const BC format. If a side is 'off', it shows NaNs.
    """
    # reuse your profile-maker: returns lambda s in [0,1] or None
    def _pf(side_name):
        side_cfg = bc.get(side_name, {"on": False})
        if not side_cfg.get("on", True):
            return None
        typ = side_cfg.get("type", "const")
        if typ == "const":
            T = float(side_cfg["T"])
            return lambda s: T
        elif typ == "gauss":
            base = float(side_cfg["base"]); amp=float(side_cfg["amp"])
            mu = float(side_cfg["mu"]); sigma=float(side_cfg["sigma"])
            sig2 = 2.0 * (sigma**2)
            return lambda s: base + amp * np.exp(-((s - mu)**2)/sig2)
        else:
            return None

    pf_left  = _pf("left")   # s along y ∈ [0,1]
    pf_right = _pf("right")  # s along y ∈ [0,1]
    pf_bot   = _pf("bottom") # s along x ∈ [0,1]
    pf_top   = _pf("top")    # s along x ∈ [0,1]

    # evaluate
    y_s = (y - y.min()) / max(Ly, 1e-9)
    x_s = (x - x.min()) / max(Lx, 1e-9)

    TL = np.array([pf_left(si)  if pf_left  else np.nan for si in y_s], dtype=float)
    TR = np.array([pf_right(si) if pf_right else np.nan for si in y_s], dtype=float)
    TB = np.array([pf_bot(si)   if pf_bot   else np.nan for si in x_s], dtype=float)
    TT = np.array([pf_top(si)   if pf_top   else np.nan for si in x_s], dtype=float)

    fig, ax = plt.subplots(2, 2, figsize=(8,6), constrained_layout=True)
    if figtitle: fig.suptitle(figtitle)

    ax[0,0].plot(y, TL, lw=2); ax[0,0].set_title("Left (x=0)")
    ax[0,1].plot(y, TR, lw=2); ax[0,1].set_title("Right (x=Lx)")
    ax[1,0].plot(x, TB, lw=2); ax[1,0].set_title("Bottom (y=0)")
    ax[1,1].plot(x, TT, lw=2); ax[1,1].set_title("Top (y=Ly)")

    for i, (t, v) in enumerate(zip(y, TL)):
        if i % 5 == 0 and not np.isnan(v):
            ax[0,0].text(t, v, f"{v:.1f}", fontsize=8, ha='center', va='bottom')
    for i, (t, v) in enumerate(zip(y, TR)):
        if i % 5 == 0 and not np.isnan(v):
            ax[0,1].text(t, v, f"{v:.1f}", fontsize=8, ha='center', va='bottom')

    for a, lbl in zip(ax.ravel(), ["y [m]","y [m]","x [m]","x [m]"]):
        a.set_xlabel(lbl); a.set_ylabel("T [K]"); a.grid(alpha=0.3)

    #plt.plot()
    if savepath: fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)


def _bc_side_summary(side_name, side_cfg):
    """Compact, human-friendly summary for one BC side."""
    if not side_cfg.get("on", True):
        return {"side": side_name, "on": False, "type": "off"}
    t = side_cfg.get("type", "const")
    if t == "const":
        return {"side": side_name, "on": True, "type": "const", "T": float(side_cfg["T"])}
    elif t == "gauss":
        return {
            "side": side_name, "on": True, "type": "gauss",
            "base": float(side_cfg["base"]),
            "amp": float(side_cfg["amp"]),
            "mu": float(side_cfg["mu"]),
            "sigma": float(side_cfg["sigma"]),
        }
    else:
        # keep generic dump
        d = {"side": side_name, "on": True, "type": t}
        d.update({k: (float(v) if isinstance(v, (int,float)) else v)
                  for k, v in side_cfg.items() if k not in ("on","type")})
        return d

def plot_case_time_stats(TT, FF, times, out_path):  
    """
    Plot min/mean/max/var vs time for both T and f (two rows).
    TT, FF: [Nt, N, N]
    """
    Nt = TT.shape[0]
    T_flat = TT.reshape(Nt, -1)
    F_flat = FF.reshape(Nt, -1)

    T_min = T_flat.min(axis=1);  T_mean = T_flat.mean(axis=1)
    T_max = T_flat.max(axis=1);  T_var  = T_flat.var(axis=1)

    f_min = F_flat.min(axis=1);  f_mean = F_flat.mean(axis=1)
    f_max = F_flat.max(axis=1);  f_var  = F_flat.var(axis=1)

    fig, ax = plt.subplots(2, 1, figsize=(8, 6), constrained_layout=True)

    # Temperature
    ax[0].plot(times, T_min,  marker="o", label="min")
    ax[0].plot(times, T_mean, marker="o", label="mean")
    ax[0].plot(times, T_max,  marker="o", label="max")
    for vals in [T_min, T_mean, T_max]:
      for t, v in zip(times, vals):
        ax[0].text(t, v, f"{v:.2f}", fontsize=8, ha='center', va='bottom')

    ax[0].set_title("Temperature stats vs time")
    ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("T [K] / var")
    ax[0].set_ylim(290,900)
    ax[0].grid(True, ls="--", alpha=0.5); ax[0].legend()

    # Liquid fraction
    ax[1].plot(times, f_min,  marker="o", label="min")
    ax[1].plot(times, f_mean, marker="o", label="mean")
    ax[1].plot(times, f_max,  marker="o", label="max")
    #ax[1].plot(times, f_var,  marker="o", label="var")
    for vals in [f_min, f_mean, f_max]:
      for t, v in zip(times, vals):
        ax[1].text(t, v, f"{v:.2f}", fontsize=8, ha='center', va='bottom')
    ax[1].set_title("Liquid fraction stats vs time")
    ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("f [-] / var")
    ax[1].grid(True, ls="--", alpha=0.5); ax[1].legend()

    #plt.plot()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

    # return stats if you want to persist them per case
    return {
        "T_min": T_min.tolist(), "T_mean": T_mean.tolist(),
        "T_max": T_max.tolist(), "T_var": T_var.tolist(),
        "f_min": f_min.tolist(), "f_mean": f_mean.tolist(),
        "f_max": f_max.tolist(), "f_var": f_var.tolist(),
    }

def plot_case_variance_stats(TT, FF, times, out_path):
    """
    Plot variance vs time for both Temperature and Liquid Fraction.
    """
    Nt = TT.shape[0]
    T_var = TT.reshape(Nt, -1).var(axis=1)
    f_var = FF.reshape(Nt, -1).var(axis=1)

    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.plot(times, T_var, marker="o", label="T variance")
    ax.plot(times, f_var, marker="o", label="f variance")

    # data labels
    for vals in [T_var, f_var]:
        for t, v in zip(times, vals):
            ax.text(t, v, f"{v:.2e}", fontsize=8, ha='center', va='bottom')

    ax.set_title("Variance vs time")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("Variance")
    ax.grid(True, ls="--", alpha=0.5)
    ax.legend()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def _timewise_stats(TT, FF, save_times):
    """
    Per-snapshot stats. Returns a list of dicts with time, T stats, f stats, and melt fractions.
    - melt_frac = fraction of cells with f>0 (any melt)
    - liquid_frac = fraction of cells with f>0.99 (near fully liquid)
    """
    out = []
    Nt = TT.shape[0]
    for k in range(Nt):
        T = TT[k]; f = FF[k]
        ncell = float(T.size)
        melt_frac  = float((f > 0.0).sum()) / ncell
        liquid_frac= float((f > 0.99).sum()) / ncell
        out.append({
            "time": float(save_times[k]),
            "T_min": float(T.min()), "T_mean": float(T.mean()), "T_max": float(T.max()),
            "f_min": float(f.min()), "f_mean": float(f.mean()), "f_max": float(f.max()),
            "melt_frac": melt_frac, "liquid_frac": liquid_frac,
        })
    return out

def write_case_stats(RUN_DIR, case_id, Q, TT, FF, save_times, bc):
    """
    Compute & append concise stats for one case to:
      - RUN_DIR/case_stats.csv    (one row per case, simple columns)
      - RUN_DIR/case_stats.jsonl  (full rich structure per case)

    Returns the full stats dict for in-memory use.
    """
    RUN_DIR = Path(RUN_DIR)
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # --- Q (heat-source) stats ---
    stats_Q = {
        "Q_min": float(Q.min()),
        "Q_mean": float(Q.mean()),
        "Q_max": float(Q.max()),
        "Q_std": float(Q.std()),
    }

    # --- BC summary (left/right/bottom/top) ---
    bc_summary = [
        _bc_side_summary("left",   bc.get("left",   {"on": False})),
        _bc_side_summary("right",  bc.get("right",  {"on": False})),
        _bc_side_summary("bottom", bc.get("bottom", {"on": False})),
        _bc_side_summary("top",    bc.get("top",    {"on": False})),
    ]

    # --- Timewise T/f stats ---
    per_time = _timewise_stats(TT, FF, save_times)

    # --- assemble full record ---
    record = {
        "case_id": int(case_id),
        "N": int(TT.shape[-1]),
        "Nt": int(TT.shape[0]),
        "times": [float(t) for t in save_times],
        "Q_stats": stats_Q,
        "BC": bc_summary,
        "per_time": per_time,
        # convenient “headline” fields for CSV:
        "T_min_first": per_time[0]["T_min"],
        "T_mean_first": per_time[0]["T_mean"],
        "T_max_last": per_time[-1]["T_max"],
        "f_mean_last": per_time[-1]["f_mean"],
        "melt_frac_last": per_time[-1]["melt_frac"],
        "liquid_frac_last": per_time[-1]["liquid_frac"],
    }

    # --- append JSONL ---
    jsonl_path = RUN_DIR / "case_stats.jsonl"
    with open(jsonl_path, "a", encoding="utf-8") as jf:
        jf.write(json.dumps(record) + "\n")

    # --- append CSV (create header if missing) ---
    csv_path = RUN_DIR / "case_stats.csv"
    csv_cols = [
        "case_id","N","Nt",
        "Q_min","Q_mean","Q_max","Q_std",
        "T_min_first","T_mean_first","T_max_last","f_mean_last","melt_frac_last","liquid_frac_last",
    ]
    make_header = not csv_path.exists()
    with open(csv_path, "a", encoding="utf-8") as cf:
        if make_header:
            cf.write(",".join(csv_cols) + "\n")
        row = [
            str(record["case_id"]),
            str(record["N"]), str(record["Nt"]),
            f'{stats_Q["Q_min"]:.6g}', f'{stats_Q["Q_mean"]:.6g}', f'{stats_Q["Q_max"]:.6g}', f'{stats_Q["Q_std"]:.6g}',
            f'{record["T_min_first"]:.6g}', f'{record["T_mean_first"]:.6g}',
            f'{record["T_max_last"]:.6g}',  f'{record["f_mean_last"]:.6g}',
            f'{record["melt_frac_last"]:.6g}', f'{record["liquid_frac_last"]:.6g}',
        ]
        cf.write(",".join(row) + "\n")

    return record

In [ ]:
def plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm, Q, qlabel="Q [W/m^3]", colormap="RdBu_r", savepath=None):
    """
    Show heat source Q plus temperature and liquid fraction snapshots.
    Rows: 0=Q (same in all columns), 1=T, 2=f
    Columns: different saved times.
    If savepath is given, saves the figure instead of showing it.
    """
    n = len(ts)
    fig, axes = plt.subplots(3, n, figsize=(4.4*n, 10), dpi=120, constrained_layout=True)

    x0, x1 = float(X.min()), float(X.max())
    y0, y1 = float(Y.min()), float(Y.max())

    # Q row 0
    for k in range(n):
        imQ = axes[0, k].imshow(Q.T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap=colormap, aspect="equal")
        axes[0, k].set_title("Heat source Q")
        axes[0, k].set_xlabel("x [m]"); axes[0, k].set_ylabel("y [m]")
        if k == 0:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85, label=qlabel)
        else:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85)

    # Temperature with T=Tm contour row 1
    for k in range(n):
        imT = axes[1, k].imshow(Ts[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="inferno", aspect="equal")
        axes[1, k].contour(X, Y, Ts[k], levels=[Tm], colors='cyan', linewidths=1.0)
        axes[1, k].set_title(f"T at t={ts[k]:.1f} s")
        axes[1, k].set_xlabel("x [m]"); axes[1, k].set_ylabel("y [m]")
        fig.colorbar(imT, ax=axes[1, k], shrink=0.85, label="T [K]")

    # Liquid fraction f row 2
    for k in range(n):
        imF = axes[2, k].imshow(Fs[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="viridis", vmin=0.0, vmax=1.0, aspect="equal")
        axes[2, k].set_title(f"Liquid fraction at t={ts[k]:.1f} s")
        axes[2, k].set_xlabel("x [m]"); axes[2, k].set_ylabel("y [m]")
        fig.colorbar(imF, ax=axes[2, k], shrink=0.85, label="f [-]")

    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [ ]:
def _downsample_1d(arr: np.ndarray, n: int) -> np.ndarray:
    if n <= 0:
        return np.empty((0,), dtype=arr.dtype)
    if arr.size == 0:
        return np.zeros((n,), dtype=np.float32)
    idx = np.linspace(0, len(arr) - 1, n).round().astype(int)
    return arr[idx].astype(np.float32)

def make_branch_BC_from_Tsnaps(TT, n_pts_per_side=S_down_BC, n_time_bc=n_time_bc) -> np.ndarray:
    """
    boundary feature = concat over selected times of [top, right, bottom, left] (downsampled)
    S_BC = 4 * n_time_bc * n_pts_per_side
    n_pts_per_side: how many points to sample per boundary side.

    n_time_bc: how many time snapshots to encode. if 3 then taken in regular 3rd interval

    If TT has 7 snapshots, n_time_bc=3, n_pts_per_side=16:
    len(t_idx)=3 , feature length = 4 * 3 * 16 = 192 (float32).
    That is your per-case branch_BC vector.

    """
    if isinstance(TT, np.ndarray) and TT.ndim == 3:
        snaps = [TT[i] for i in range(TT.shape[0])]
    else:
        snaps = list(TT)

    Nt = len(snaps)
    if Nt == 0:
        return np.zeros((4 * n_time_bc * n_pts_per_side,), dtype=np.float32)

    if n_time_bc >= Nt:
        t_idx = np.arange(Nt)
    else:
        t_idx = np.linspace(0, Nt - 1, n_time_bc).round().astype(int)

    pieces = []
    for k in t_idx:
        T = snaps[k]                  # [N, N]
        N = T.shape[0]
        top    = _downsample_1d(T[0,      :], n_pts_per_side)
        right  = _downsample_1d(T[:,  N-1], n_pts_per_side)
        bottom = _downsample_1d(T[N-1,   :], n_pts_per_side)
        left   = _downsample_1d(T[:,      0], n_pts_per_side)
        pieces.extend([*top, *right, *bottom, *left])
    return np.asarray(pieces, dtype=np.float32)

def _ensure_full_Q(Q: np.ndarray, N: int) -> np.ndarray:
    """If Q isn't (N,N), upsample with separable 1D np.interp to avoid broadcast errors in solver."""
    Q = np.asarray(Q, dtype=np.float32)
    if Q.shape == (N, N):
        return Q
    qN_y, qN_x = Q.shape
    xq = np.linspace(0.0, 1.0, qN_x, dtype=np.float32)
    yq = np.linspace(0.0, 1.0, qN_y, dtype=np.float32)
    x  = np.linspace(0.0, 1.0, N,    dtype=np.float32)
    y  = np.linspace(0.0, 1.0, N,    dtype=np.float32)
    # interp along x for each row
    Qx = np.stack([np.interp(x, xq, row) for row in Q], axis=0)           # [qN_y, N]
    # interp along y for each column
    Q_full = np.stack([np.interp(y, yq, Qx[:, j]) for j in range(N)], axis=1)  # [N, N]
    return Q_full.astype(np.float32)

def make_branch_Q_times(Q, Nt, sensor_mode="full", S_down=40):
    """
    Return branch_Q concatenated across Nt snapshots.
    - If Q is a single 2D array: repeats the same vector Nt times.
    - If Q is a list/tuple of 2D arrays (length Nt): encodes each and concatenates.
    Shape: [S_Q * Nt]
    """
    def enc(oneQ):
        if sensor_mode == "full":
            return sensorize_field(oneQ, mode="full")
        else:
            return sensorize_field(oneQ, mode="downsample", S=S_down)

    if isinstance(Q, (list, tuple)):
        assert len(Q) == Nt, "If Q is time-varying you must pass Nt maps."
        vecs = [enc(Qk) for Qk in Q]
    else:
        v = enc(Q)
        vecs = [v] * Nt
    return np.concatenate(vecs, axis=0).astype(np.float32)


In [ ]:
import time, json
from pathlib import Path
import numpy as np

def build_deeponet_dataset():
    """
    Saves (with separated branch & trunk streams):
      - deeponet_temp_dataset.npz
          branch_Q:   [C, S_Q]
          branch_BC:  [C, S_BC]
          trunk_xy:   [M_total, 2]   (meters)
          trunk_t:    [M_total, 1]   (seconds)
          yT:         [M_total, 1]
          case_ids:   [M_total]
          meta:       dict

      - deeponet_frac_dataset.npz
          branch_T:   [C, S_T]   (concat of T snapshots per case)
          trunk_xy:   [M_total, 2]
          trunk_t:    [M_total, 1]
          yf:         [M_total, 1]
          case_ids:   [M_total]
          meta:       dict

      - case_stats.csv / case_stats.jsonl (per-case stats, human-friendly)
      - cases_bc.json (raw BC dict per case)
      - splits.json   (train/val/test splits by case)
    Uses your global parameters (as you posted).
    """

    # ----------------- pull globals -----------------
    N        = int(globals()["N"])
    Lx       = float(globals()["Lx"])
    Ly       = float(globals()["Ly"])
    rho      = float(globals()["rho"])
    cp       = float(globals()["cp"])
    k        = float(globals()["k"])
    L_lat    = float(globals()["L_lat"])
    T_m      = float(globals().get("T_m", globals().get("Tm")))
    T_init   = float(globals()["T_init"])
    T_bound  = float(globals()["T_bound"])
    t_end    = float(globals()["t_end"])
    save_times = tuple(globals()["save_times"])
    cfl      = float(globals()["cfl"])

    # GP / Q field params
    q_scale        = float(globals()["q_scale"])
    Q_length_scale = float(globals()["Q_length_scale"])
    Q_sigma        = float(globals()["Q_sigma"])

    # dataset sizes
    NUM_CASES      = int(globals()["NUM_CASES"])
    TRAIN_FRAC     = float(globals()["TRAIN_FRAC"])
    VAL_FRAC       = float(globals()["VAL_FRAC"])

    # DeepONet sampling
    sensor_mode    = str(globals()["sensor_mode"])
    S_down         = int(globals()["S_down"])
    per_time       = int(globals()["points_per_case_per_time"])
    n_time_bc      = int(globals()["n_time_bc"])
    S_down_BC      = int(globals()["S_down_BC"])
    
    # boundary variety
    only_lr_vary   = bool(globals()["only_lr_vary"])
    All_side_const_temp_boundary = bool(globals()["All_side_const_temp_boundary"])

    # output dir
    RUN_DIR        = Path(globals()["RUN_DIR"])
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # BC feature density
    bc_pts_per_side   = int(globals().get("bc_pts_per_side", S_down_BC))
    bc_time_snapshots = int(globals().get("bc_time_snapshots", n_time_bc))

    # header
    print("────────────────────────────────────────────────────────")
    print("Building DeepONet datasets (separate Q/BC + XY/T trunks)")
    print(f"N={N}, Lx={Lx}, Ly={Ly}, times={list(save_times)}, T_bound={T_bound}")
    print(f"sensor_mode={sensor_mode}, S_down={S_down}, per_time={per_time}")
    print(f"NUM_CASES={NUM_CASES}, TRAIN_FRAC={TRAIN_FRAC}, VAL_FRAC={VAL_FRAC}")
    print(f"q_scale={q_scale}, Q_length_scale={Q_length_scale}, Q_sigma={Q_sigma}")
    print(f"BC feature: pts/side={bc_pts_per_side}, time_snaps={bc_time_snapshots}")
    print(f"Output dir: {RUN_DIR.resolve()}")
    print("────────────────────────────────────────────────────────")

    # ----------------- containers -----------------
    branch_Q_list   = []   # [C, S_Q]
    branch_T_list   = []   # [C, S_T]
    branch_BC_list  = []   # [C, S_BC]
    trunk_xy_all    = []   # [M_total, 2]
    trunk_t_all     = []   # [M_total, 1]
    target_T_all    = []   # [M_total, 1]
    target_f_all    = []   # [M_total, 1]
    case_id_all     = []   # [M_total]
    case_meta       = []   # list of dicts (for cases_bc.json)

    # ----------------- build cases -----------------
    for i in range(1, NUM_CASES + 1):
        # 1) BC dict for this case
        bc = make_bc_case(i, T_bound=T_bound, only_lr=only_lr_vary, all_const=All_side_const_temp_boundary)

        # 2) Heat source Q from GP posterior
        genQ = HeatSource2DRBFPosterior(
            grid_size=N, length=1.0, length_scale=Q_length_scale,
            sigma=Q_sigma, jitter=1e-6, seed=10_000 + i
        )
        Q_dimless = genQ.sample_posterior(
            enforce_zero_mean=False,
            force_sign='positive',
            target_mean=1.0,
            keep_boundary_zero=False
        )
        Q = _ensure_full_Q(q_scale * Q_dimless, N).astype(np.float32)  # [N,N]

        # 3) Simulate PCM with source
        Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
            Q_Wm3=Q, nx=N, ny=N, Lx=Lx, Ly=Ly,
            rho=rho, cp=cp, k=k, L_lat=L_lat,
            T_init=T_init, Tm=T_m,
            t_end=t_end, cfl=cfl, save_times=save_times, bc=bc
        )
        TT = np.asarray(Ts, dtype=np.float32)  # [Nt, N, N]
        FF = np.asarray(Fs, dtype=np.float32)  # [Nt, N, N]

        # 4) Branch features (separate)

        Q_branch = make_branch_Q_times(Q, Nt=len(save_times), sensor_mode=sensor_mode, S_down=S_down)
        if sensor_mode == "full":
            #Q_branch = sensorize_field(Q, mode="full")                 # [N*N]
            T_branch = np.concatenate([sensorize_field(TT[k], "full")
                                       for k in range(len(save_times))], axis=0)
        else:
            #Q_branch = sensorize_field(Q, mode="downsample", S=S_down) # [S_down*S_down]
            T_branch = np.concatenate([sensorize_field(TT[k], "downsample", S=S_down)
                                       for k in range(len(save_times))], axis=0)

        BC_branch = make_branch_BC_from_Tsnaps(
            TT, n_pts_per_side=bc_pts_per_side, n_time_bc=bc_time_snapshots
        )  # [4 * n_time_bc * n_pts_per_side]

        branch_Q_list.append(Q_branch)
        branch_T_list.append(T_branch)
        branch_BC_list.append(BC_branch)

        # 5) Trunk sampling (split XY / T)
        rng_case = np.random.default_rng(12345 + i)
        xy, tcol, ij, tids = sample_trunk_points_split(
            N, Lx, Ly, save_times, per_time, rng_case
        )
        trunk_xy_all.append(xy)      # [M_case, 2]
        trunk_t_all.append(tcol)     # [M_case, 1]

        # 6) Targets matched to those points
        #    For each time block (ti), use TT[ti] with ij rows in that block
        yT_case = []
        yf_case = []
        for ti in range(len(save_times)):
            s, e = ti * per_time, (ti + 1) * per_time
            jj = ij[s:e, 0]  # row index (y)
            ii = ij[s:e, 1]  # col index (x)
            yT_case.append(TT[ti][jj, ii])
            yf_case.append(FF[ti][jj, ii])
        yT_case = np.concatenate(yT_case, axis=0).astype(np.float32)[:, None]
        yf_case = np.concatenate(yf_case, axis=0).astype(np.float32)[:, None]
        target_T_all.append(yT_case)
        target_f_all.append(yf_case)

        # 7) Case ids
        M_case = per_time * len(save_times)
        case_id_all.append(np.full((M_case,), i-1, dtype=np.int64))

        # 8) Stats logging (CSV + JSONL)
        _ = write_case_stats(
            RUN_DIR=RUN_DIR, case_id=i, Q=Q, TT=TT, FF=FF,
            save_times=save_times, bc=bc
        )
        case_meta.append({"case_id": i, "bc": bc})  # raw BC dump

        # ---- concise per-case log ----
        print(f"[case {i:>2d}/{NUM_CASES}] "
              f"Q={Q.shape}  TT={TT.shape}  M_case={M_case}  "
              f"S_Q={Q_branch.shape[0]}  S_T={T_branch.shape[0]}  S_BC={BC_branch.shape[0]}")
        
        # Use your existing plotting function (grid of columns per time)
        plot_fields_with_Q(
            Ts=TT, Fs=FF, ts=save_times, X=X, Y=Y, Tm=T_m,
            Q=Q, qlabel="Q [W/m^3]", colormap="RdBu_r",
            savepath=RUN_DIR / f"case_{i:02d}_temp_and_liquid_profiles.png"
        )

        # B) Boundary profiles (lines for each side)
        xs = np.linspace(0.0, Lx, N, dtype=np.float32)
        ys = np.linspace(0.0, Ly, N, dtype=np.float32)
        plot_boundary_profiles(
            bc=bc, x=xs, y=ys, Lx=Lx, Ly=Ly,
            figtitle=f"Case {i} boundary profiles",
            savepath=RUN_DIR / f"case_{i:02d}_boundary_profiles.png"
        )
        
        # C) Time-series: min/mean/max/var vs time for T and f
        _ = plot_case_time_stats(
            TT=TT, FF=FF, times=save_times,
            out_path=RUN_DIR / f"case_{i:02d}_time_stats.png"
        )
        _ = plot_case_variance_stats(TT, FF, times=save_times, out_path=RUN_DIR / f"case_{i:02d}_variance_stats.png")

    # ----------------- stack & save -----------------
    branch_Q   = np.vstack(branch_Q_list).astype(np.float32)  # [C, S_Q]
    branch_T   = np.vstack(branch_T_list).astype(np.float32)  # [C, S_T]
    branch_BC  = np.vstack(branch_BC_list).astype(np.float32) # [C, S_BC]
    trunk_xy   = np.vstack(trunk_xy_all).astype(np.float32)   # [M_total, 2]
    trunk_t    = np.vstack(trunk_t_all).astype(np.float32)    # [M_total, 1]
    yT         = np.vstack(target_T_all).astype(np.float32)   # [M_total, 1]
    yf         = np.vstack(target_f_all).astype(np.float32)   # [M_total, 1]
    case_ids   = np.concatenate(case_id_all, axis=0).astype(np.int64)

    meta = dict(
        N=N, Lx=Lx, Ly=Ly, save_times=list(save_times),
        sensor_mode=sensor_mode, S_down=S_down,
        only_lr_vary=only_lr_vary,
        S_BC=int(branch_BC.shape[1]),
        points_per_case_per_time=per_time,
        desc="Separated branch_Q/branch_BC and trunk_xy/trunk_t for 4-network dot-product DeepONet"
    )

    # splits
    rng = np.random.default_rng(2024)
    all_cases = np.arange(NUM_CASES, dtype=int)
    rng.shuffle(all_cases)
    n_train = int(round(NUM_CASES * TRAIN_FRAC))
    n_val   = int(round(NUM_CASES * VAL_FRAC))
    split = {
        "train": all_cases[:n_train].tolist(),
        "val":   all_cases[n_train:n_train+n_val].tolist(),
        "test":  all_cases[n_train+n_val:].tolist(),
    }

    # save artifacts
    np.savez_compressed(
        RUN_DIR / "deeponet_temp_dataset.npz",
        branch_Q=branch_Q,
        branch_BC=branch_BC,
        trunk_xy=trunk_xy,
        trunk_t=trunk_t,
        yT=yT,
        case_ids=case_ids,
        meta=meta,
    )
    np.savez_compressed(
        RUN_DIR / "deeponet_frac_dataset.npz",
        branch_T=branch_T,
        trunk_xy=trunk_xy,
        trunk_t=trunk_t,
        yf=yf,
        case_ids=case_ids,
        meta=meta,
    )
    with open(RUN_DIR / "cases_bc.json", "w") as f:
        json.dump(case_meta, f, indent=2)
    with open(RUN_DIR / "splits.json", "w") as f:
        json.dump(split, f, indent=2)

    # ----------------- summary -----------------
    M_total = trunk_xy.shape[0]
    print("────────────────────────────────────────────────────────")
    print("Finished dataset build (separate branches & trunks)")
    print(f"Cases: {NUM_CASES} | Train/Val/Test: {len(split['train'])}/{len(split['val'])}/{len(split['test'])}")
    print(f"Branch shapes:  Q{branch_Q.shape}  T{branch_T.shape}  BC{branch_BC.shape}")
    print(f"Trunk shapes:   XY{trunk_xy.shape}  T{trunk_t.shape}")
    print(f"Targets:        yT{yT.shape}  yf{yf.shape}")
    print(f"Trunk points total: {M_total}  (~{M_total//NUM_CASES} per case)")
    print("Saved:")
    print(f"  {RUN_DIR/'deeponet_temp_dataset.npz'}")
    print(f"  {RUN_DIR/'deeponet_frac_dataset.npz'}")
    print(f"  {RUN_DIR/'case_stats.csv'}")
    print(f"  {RUN_DIR/'case_stats.jsonl'}")
    print(f"  {RUN_DIR/'cases_bc.json'}")
    print(f"  {RUN_DIR/'splits.json'}")
    print("────────────────────────────────────────────────────────")

    return RUN_DIR


In [ ]:
RUN_DIR = build_deeponet_dataset()

DATA ANALYSIS

In [ ]:
temp=r"runs/dataset_run_20251112-182232/deeponet_temp_dataset.npz"
npz=np.load(temp,allow_pickle=True)
print("All keys in temp file are: ")
print(list(npz.keys()))

temp1=r"runs/dataset_run_20251112-182232/deeponet_frac_dataset.npz"
npz_frac=np.load(temp1,allow_pickle=True)
print("All keys in frac.npz file are: ")
print(list(npz_frac.keys()))

print(f" Branch_Q: {npz["branch_Q"].shape}")
print(f" Branch_BC: {npz["branch_BC"].shape}")
print(f" yf shape: {npz_frac["yf"].shape}")
print(f" Branch_T: {npz_frac["branch_T"].shape}")
print(f" meta of temp: {npz["meta"]}")
print(f" meta of liq frac: {npz_frac["meta"]}")


In [ ]:
# ==========================================
# Single-plots: per-case series for mean/min/max/var (T & f)
# ==========================================

dataset_dir = r"runs/dataset_run_20251112-182232"
temp_npz = os.path.join(dataset_dir, "deeponet_temp_dataset.npz")
frac_npz = os.path.join(dataset_dir, "deeponet_frac_dataset.npz")

def _ensure(cond, msg):
    if not cond: raise RuntimeError(msg)

_ensure(os.path.isfile(temp_npz), f"Missing: {temp_npz}")
_ensure(os.path.isfile(frac_npz), f"Missing: {frac_npz}")

Tz = np.load(temp_npz, allow_pickle=True)
Fz = np.load(frac_npz, allow_pickle=True)

# Required keys in your point datasets:
for k in ["trunk_t","yT","case_ids"]:
    _ensure(k in Tz, f"Key '{k}' not found in {temp_npz}")

trunk_t_T = Tz["trunk_t"].squeeze().astype(float)
yT        = Tz["yT"].squeeze().astype(float)
case_T    = Tz["case_ids"].squeeze()

# Frac keys (support variants)
yF = Fz["yf"].squeeze().astype(float)

t_F = Fz["trunk_t"].squeeze().astype(float) if "trunk_t" in Fz else trunk_t_T
case_F = Fz["case_ids"].squeeze() if "case_ids" in Fz else case_T

# Helper: stable unique times (collapse float jitter)
def unique_times_stable(t, decimals=9):
    t = np.round(np.asarray(t, float), decimals=decimals)
    return np.unique(t)

# Per-case time series from (x,y,t) points: mean/min/max/var across spatial points at each t
def per_case_series(case_id, t_raw, y_scalar):
    m = (case_id == case_ids)   # will set case_ids before calls
    t = t_raw[m]; y = y_scalar[m]
    tr = np.round(t, 9)
    ut = np.unique(tr)
    K = len(ut)
    mean = np.empty(K); var = np.empty(K); mn = np.empty(K); mx = np.empty(K)
    for i, tt in enumerate(ut):
        yy = y[tr == tt]
        mean[i] = yy.mean()
        var[i]  = yy.var(ddof=1) if yy.size > 1 else 0.0
        mn[i]   = yy.min()
        mx[i]   = yy.max()
    return ut, dict(mean=mean, var=var, min=mn, max=mx)

# Build union time axes (temperature drives the reference axis)
U_times_T = unique_times_stable(trunk_t_T)
U_times_F = unique_times_stable(t_F)

# Cases present in both files
cases_T = pd.unique(case_T)
cases_F = pd.unique(case_F)
common_cases = np.intersect1d(cases_T, cases_F)

# Compute per-case series for T
case_ids = case_T  # used inside per_case_series
T_series = {}
for c in common_cases:
    ut, s = per_case_series(c, trunk_t_T, yT)
    T_series[int(c)] = (ut, s)

# Compute per-case series for f
case_ids = case_F
F_series = {}
for c in common_cases:
    ut, s = per_case_series(c, t_F, yF)
    F_series[int(c)] = (ut, s)

# Align any per-case series to a common axis (nearest exact match on save_times)
def align_to_axis(ut, arr, ref):
    out = np.full_like(ref, np.nan, dtype=float)
    pos = {float(v): i for i,v in enumerate(ut)}
    for i, tval in enumerate(ref):
        j = pos.get(float(tval), None)
        if j is not None: out[i] = arr[j]
    return out

# Build aligned matrices (n_cases x K) for each metric
def build_matrix(series_dict, ref_times, key):
    ids = sorted(series_dict.keys())
    M = np.empty((len(ids), len(ref_times)), dtype=float)
    M[:] = np.nan
    for r, cid in enumerate(ids):
        ut, s = series_dict[cid]
        M[r] = align_to_axis(ut, s[key], ref_times)
    return ids, M

idsT, T_mean = build_matrix(T_series, U_times_T, "mean")
_,    T_min  = build_matrix(T_series, U_times_T, "min")
_,    T_max  = build_matrix(T_series, U_times_T, "max")
_,    T_var  = build_matrix(T_series, U_times_T, "var")

idsF, F_mean = build_matrix(F_series, U_times_F, "mean")
_,    F_min  = build_matrix(F_series, U_times_F, "min")
_,    F_max  = build_matrix(F_series, U_times_F, "max")
_,    F_var  = build_matrix(F_series, U_times_F, "var")

# -------- plotting helpers (no huge legends) --------
def _multi_lines(ref_times, M, title, ylabel, legend=False):
    plt.figure(figsize=(8,5))
    for r in range(M.shape[0]):
        plt.plot(ref_times, M[r], lw=1)
    plt.title(title); plt.xlabel("time (s)"); plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    if legend:
        # Caution: 50 labels can clutter; use only if you truly want it
        labels = [f"Case {cid}" for cid in idsT]  # or idsF depending on matrix
        plt.legend(labels, fontsize=7, ncol=3)
    plt.tight_layout()

# -------- Temperature plots (all 50 cases on one plot each) --------
_multi_lines(U_times_T, T_mean, "Mean Temperature across cases", "T_mean (K)")
_multi_lines(U_times_T, T_max,  "Max Temperature across cases",  "T_max (K)")
_multi_lines(U_times_T, T_min,  "Min Temperature across cases",  "T_min (K)")
_multi_lines(U_times_T, T_var,  "Temperature Variance across cases", "Var(T)")

# -------- Liquid fraction plots --------
_multi_lines(U_times_F, F_mean, "Mean Liquid Fraction across cases", "f_mean (-)")
_multi_lines(U_times_F, F_max,  "Max Liquid Fraction across cases",  "f_max (-)")
_multi_lines(U_times_F, F_min,  "Min Liquid Fraction across cases",  "f_min (-)")
_multi_lines(U_times_F, F_var,  "Liquid Fraction Variance across cases", "Var(f)")


In [ ]:
# ================================================================
# Trainability Diagnostics: PCA + Histograms + Correlations + Verdict
# ===============================================================

# ------------------- configuration -------------------
dataset_dir = r"runs/dataset_run_20251112-182232"
temp_npz = os.path.join(dataset_dir, "deeponet_temp_dataset.npz")
frac_npz = os.path.join(dataset_dir, "deeponet_frac_dataset.npz")
cases_bc_json = os.path.join(dataset_dir, "cases_bc.json")  # list with {"case_id":..,"BC":"{...}"}

def _ensure(cond, msg):
    if not cond: raise RuntimeError(msg)

# ------------------- load files -------------------
_ensure(os.path.isfile(temp_npz), f"Missing: {temp_npz}")
_ensure(os.path.isfile(frac_npz), f"Missing: {frac_npz}")
_ensure(os.path.isfile(cases_bc_json), f"Missing: {cases_bc_json}")

Tz = np.load(temp_npz, allow_pickle=True)
Fz = np.load(frac_npz, allow_pickle=True)

# Required in temp npz (point dataset)
for k in ["trunk_t","yT","case_ids","branch_Q","branch_BC"]:
    _ensure(k in Tz, f"Key '{k}' not found in {temp_npz}")

t_T      = Tz["trunk_t"].squeeze().astype(float)    # (Npts,)
yT       = Tz["yT"].squeeze().astype(float)         # (Npts,)
case_T   = Tz["case_ids"].squeeze()                 # (Npts,)
bQ_all   = Tz["branch_Q"]                           # (Npts, S_Q)
bBC_all  = Tz["branch_BC"]                          # (Npts, S_BC)
meta_T   = Tz["meta"].item() if "meta" in Tz else {}

# Frac keys (variants supported)
if   "yF"   in Fz: yF = Fz["yF"].squeeze().astype(float)
elif "yf"   in Fz: yF = Fz["yf"].squeeze().astype(float)
else:              yF = Fz["frac"].squeeze().astype(float)
t_F      = Fz["trunk_t"].squeeze().astype(float) if "trunk_t" in Fz else t_T
case_F   = Fz["case_ids"].squeeze() if "case_ids" in Fz else case_T
meta_F   = Fz["meta"].item() if "meta" in Fz else {}

# ------------------- utilities -------------------
def recover_times(raw_t, meta):
    raw_t = np.asarray(raw_t, float)
    if isinstance(meta, dict) and {"t_min","t_max"} <= set(meta.keys()):
        tmin, tmax = float(meta["t_min"]), float(meta["t_max"])
        if raw_t.max() <= 1.0 + 1e-8 and raw_t.min() >= -1e-8:
            return tmin + raw_t*(tmax - tmin)
    return raw_t

def unique_times_stable(t, decimals=9):
    t = np.round(np.asarray(t, float), decimals=decimals)
    return np.unique(t)

def per_case_series(case_id, t_raw, y_scalar, case_ids):
    m = (case_id == case_ids)
    t = t_raw[m]; y = y_scalar[m]
    tr = np.round(t, 9)
    ut = np.unique(tr)
    K = len(ut)
    mean = np.empty(K); var = np.empty(K); mn = np.empty(K); mx = np.empty(K)
    for i, tt in enumerate(ut):
        yy = y[tr == tt]
        mean[i] = yy.mean()
        var[i]  = yy.var(ddof=1) if yy.size > 1 else 0.0
        mn[i]   = yy.min()
        mx[i]   = yy.max()
    return ut, dict(mean=mean, var=var, min=mn, max=mx)

def align_to_axis(ut, arr, ref_times):
    out = np.full_like(ref_times, np.nan, dtype=float)
    pos = {float(v): i for i,v in enumerate(ut)}
    for i, tval in enumerate(ref_times):
        j = pos.get(float(tval), None)
        if j is not None: out[i] = arr[j]
    return out

# ------------------- prep per-case time series -------------------
t_T_phys = recover_times(t_T, meta_T)
t_F_phys = recover_times(t_F, meta_F)

U_times_T = unique_times_stable(t_T_phys)
U_times_F = unique_times_stable(t_F_phys)

cases = np.intersect1d(pd.unique(case_T), pd.unique(case_F))
cases = np.asarray(sorted(cases), dtype=int)

T_series = {}
F_series = {}
for cid in cases:
    ut, Ts = per_case_series(cid, t_T_phys, yT, case_T)
    uf, Fs = per_case_series(cid, t_F_phys, yF, case_F)
    T_series[cid] = (ut, Ts)
    F_series[cid] = (uf, Fs)

def build_matrix(series_dict, ref_times, key):
    ids = sorted(series_dict.keys())
    M = np.empty((len(ids), len(ref_times)), dtype=float); M[:] = np.nan
    for r, cid in enumerate(ids):
        ut, s = series_dict[cid]
        M[r] = align_to_axis(ut, s[key], ref_times)
    return np.asarray(ids, int), M

idsT, T_mean = build_matrix(T_series, U_times_T, "mean")
_,    T_min  = build_matrix(T_series, U_times_T, "min")
_,    T_max  = build_matrix(T_series, U_times_T, "max")
_,    T_var  = build_matrix(T_series, U_times_T, "var")

idsF, F_mean = build_matrix(F_series, U_times_F, "mean")
_,    F_min  = build_matrix(F_series, U_times_F, "min")
_,    F_max  = build_matrix(F_series, U_times_F, "max")
_,    F_var  = build_matrix(F_series, U_times_F, "var")

# ------------------- per-case inputs: mean Q & max Q (from branch_Q) -------------------
# branch_Q repeats per point; we compress to 1 vector per case via median across points, then take mean/max
npts_points  = case_T.shape[0]
nuniq_cases  = len(cases)
rows_bQ      = bQ_all.shape[0]

# helper: map case id -> row in a sorted order (for per-case layout)
_sorted_cases = np.array(sorted(cases), dtype=int)
_case_to_row  = { int(cid): i for i, cid in enumerate(_sorted_cases) }

def get_bQ_rows_for_case(cid: int):
    """
    Returns an array of shape (M, S_Q) containing the branch_Q rows that belong to this case.
    Works whether branch_Q is stored per-point or per-case.
    """
    if rows_bQ == npts_points:
        # per-point layout: filter by the point-level case mask
        return bQ_all[case_T == cid]                     # (npts_in_case, S_Q)
    elif rows_bQ == nuniq_cases:
        # per-case layout: one row per unique case, in the same sorted order we use
        r = _case_to_row[int(cid)]
        return bQ_all[r:r+1]                             # (1, S_Q)
    elif rows_bQ >= (np.max(cases) + 1):
        # indexed by numeric case id directly (0..max_cid); common alternative
        return bQ_all[int(cid):int(cid)+1]               # (1, S_Q)
    else:
        raise RuntimeError(
            f"Unrecognized branch_Q layout: rows={rows_bQ}, "
            f"expected {npts_points} (per-point) or {nuniq_cases} (per-case)"
        )

case_mean_Q, case_max_Q = [], []
for cid in cases:
    BQ_case_rows = get_bQ_rows_for_case(int(cid))        # (M, S_Q)
    # robust collapse to a single vector per case
    BQ_case = np.median(BQ_case_rows, axis=0)            # (S_Q,)
    case_mean_Q.append(float(np.mean(BQ_case)))
    case_max_Q.append(float(np.max(BQ_case)))

case_mean_Q = np.asarray(case_mean_Q, float)
case_max_Q  = np.asarray(case_max_Q,  float)

# ------------------- per-case boundary amplitude from cases_bc.json -------------------
with open(cases_bc_json, "r") as f:
    bc_list = json.load(f)   # list of dicts: {"case_id":..., "BC":"{...}"}
def _bc_amp(bc_obj):
    # amplitude signal from BC: sum of |amp| on any 'gauss' edges; if const only, amp=0
    s = 0.0; n = 0
    for edge in ["left","right","top","bottom"]:
        if edge in bc_obj and isinstance(bc_obj[edge], dict) and bc_obj[edge].get("on", False):
            t = bc_obj[edge].get("type","const")
            if t == "gauss":
                s += abs(float(bc_obj[edge].get("amp", 0.0)))
                n += 1
    return s if n>0 else 0.0

# build mapping case_id -> amplitude
case_bc_amp = {}
for item in bc_list:
    if not isinstance(item, dict): continue
    cid = int(item.get("case_id"))
    bc_raw = item.get("BC")
    try:
        bc_obj = bc_raw if isinstance(bc_raw, dict) else ast.literal_eval(bc_raw)
    except Exception:
        bc_obj = {}
    case_bc_amp[cid] = _bc_amp(bc_obj)

bc_amp = np.array([case_bc_amp.get(int(cid), 0.0) for cid in cases], dtype=float)

# ------------------- PCA on per-case mean Temperature curves -------------------
# Standardize across time (per column) to remove scale differences across times
X = np.nan_to_num(T_mean)  # (n_cases, K)
X = X[~np.isnan(X).all(axis=1)]  # drop all-NaN rows if any
X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

n_comp = min(X_std.shape[0], X_std.shape[1])
pca = PCA(n_components=n_comp, random_state=0)
pca.fit(X_std)
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100.0

# find PCs to reach 90%
n90 = int(np.searchsorted(cumvar, 90.0) + 1)

# ------------------- Plots -------------------
# 1) PCA variance curve
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(cumvar)+1), cumvar, marker='o')
plt.axhline(90, ls='--')
plt.title("PCA: Cumulative Variance Explained (T_mean per case)")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative variance (%)")
plt.grid(True); plt.tight_layout()

# 2) PCA scatter (PC1 vs PC2)
proj = pca.transform(X_std)
plt.figure(figsize=(5,5))
plt.scatter(proj[:,0], proj[:,1], s=30)
plt.title("Cases on first 2 PCA components (T_mean)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.grid(True); plt.tight_layout()

# 3) Histograms: mean Q, max Q, BC amplitude
plt.figure(figsize=(12,3.8))
plt.subplot(1,3,1); plt.hist(case_mean_Q, bins=12); plt.title("Histogram: per-case mean Q"); plt.xlabel("mean(Q)"); plt.ylabel("count"); plt.grid(True, alpha=0.3)
plt.subplot(1,3,2); plt.hist(case_max_Q, bins=12);  plt.title("Histogram: per-case max Q");  plt.xlabel("max(Q)");  plt.grid(True, alpha=0.3)
plt.subplot(1,3,3); plt.hist(bc_amp, bins=12);      plt.title("Histogram: BC amplitude sum"); plt.xlabel("Σ|amp|"); plt.grid(True, alpha=0.3)
plt.tight_layout()

# 4) Correlations with final temperature (last valid time per case)
final_idx = ~np.isnan(T_mean).all(axis=0)
if final_idx.any():
    last_k = np.where(final_idx)[0][-1]
else:
    last_k = -1
final_T_mean = np.nan_to_num(T_mean[:, last_k])

def safe_pearson_r(x, y, min_pts=3):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < min_pts:
        return np.nan, "Not enough valid points for correlation."
    xs, ys = x[m], y[m]
    if np.allclose(xs, xs[0]) or np.allclose(ys, ys[0]):
        return np.nan, "Zero variance in one variable; correlation undefined."
    xm = xs - xs.mean()
    ym = ys - ys.mean()
    denom = np.sqrt((xm**2).sum()) * np.sqrt((ym**2).sum())
    if denom <= 1e-12:
        return np.nan, "Denominator too small; correlation unstable."
    return float((xm*ym).sum() / denom), ""

def scatter_with_fit(x, y, title, xlabel, ylabel, min_pts=3):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    xs, ys = x[m], y[m]

    plt.figure(figsize=(5.5,4.2))
    plt.scatter(xs, ys, s=20)
    added = False

    # Only attempt fit if we have enough points and non-zero variance in x and y
    if len(xs) >= min_pts and (np.ptp(xs) > 1e-12) and (np.ptp(ys) > 1e-12):
        try:
            mfit, bfit = np.polyfit(xs, ys, 1)
            xx = np.linspace(xs.min(), xs.max(), 100)
            plt.plot(xx, mfit*xx + bfit)
            added = True
        except np.linalg.LinAlgError:
            # leave without a fit line; annotate instead
            added = False

    if not added:
        # annotate why no line is shown
        if len(xs) < min_pts:
            note = f"no fit: only {len(xs)} valid points"
        elif np.ptp(xs) <= 1e-12:
            note = "no fit: near-constant x"
        elif np.ptp(ys) <= 1e-12:
            note = "no fit: near-constant y"
        else:
            note = "no fit: least-squares failed"
        plt.annotate(note, xy=(0.02, 0.95), xycoords="axes fraction")

    plt.title(title); plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.grid(True); plt.tight_layout()

r_q,  msg_q  = safe_pearson_r(case_mean_Q, final_T_mean)
r_qm, msg_qm = safe_pearson_r(case_max_Q,  final_T_mean)
r_bc, msg_bc = safe_pearson_r(bc_amp,      final_T_mean)

scatter_with_fit(case_mean_Q, final_T_mean, f"Mean Q vs Final T (r={r_q if np.isfinite(r_q) else 'NA'})",
                 "per-case mean Q", "final T_mean")
scatter_with_fit(case_max_Q,  final_T_mean, f"Max Q vs Final T (r={r_qm if np.isfinite(r_qm) else 'NA'})",
                 "per-case max Q", "final T_mean")
scatter_with_fit(bc_amp,      final_T_mean, f"BC amplitude vs Final T (r={r_bc if np.isfinite(r_bc) else 'NA'})",
                 "Σ|BC amp|", "final T_mean")

def corr_text(r, xname, extra=""):
    if not np.isfinite(r):
        return f"Correlation({xname}, final T) not available. {extra}".strip()
    trend = "positive" if r > 0 else "negative"
    mag = abs(r)
    if   mag >= 0.7: lvl = "strong"
    elif mag >= 0.4: lvl = "moderate"
    elif mag >= 0.2: lvl = "weak"
    else:            lvl = "very weak / none"
    return f"{lvl} {trend} correlation (r={r:.2f}) between {xname} and final mean temperature."

print("• Input→Output relationships:")
print("  -", corr_text(r_q,  "per-case mean Q", msg_q))
print("  -", corr_text(r_qm, "per-case max Q",  msg_qm))
print("  -", corr_text(r_bc, "BC amplitude sum", msg_bc))

# ------------------- Plain-English Interpretation -------------------

print("\n================ PCA INTERPRETATION (Simple English) ================")
if len(cumvar) >= 1:
    print(f"• PC1 alone explains ~{cumvar[0]:.1f}% of variance in your per-case mean temperature curves.")
if len(cumvar) >= 2:
    print(f"• PC1+PC2 explain ~{cumvar[1]:.1f}%.")
if len(cumvar) >= 3:
    print(f"• PC1+PC2+PC3 explain ~{cumvar[2]:.1f}%.")

print(f"• Number of PCs needed to reach 90% variance: n90 = {n90}")

# Step-4 rules
if len(cumvar) >= 3 and cumvar[2] >= 85.0:
    verdict_pca = "EXCELLENT: First 1–3 PCs already capture ≥85% variance. The dataset has structured variation and should be very learnable."
elif n90 <= 10:
    verdict_pca = "GOOD: You need 5–10 PCs to reach 90%. This is still fine; there is meaningful structure, though variation may be a bit more diverse."
elif n90 > 15:
    verdict_pca = "WEAK: Needing more than 15 PCs suggests the variation is very spread out or noisy; training may be harder and generalization weaker."
else:
    verdict_pca = "BORDERLINE: Variation is present but not strongly concentrated; model can still learn with careful regularization and sufficient data."

print(f"⇒ PCA verdict: {verdict_pca}")

print("\n================ COMPLEMENTARY CHECKS (Simple English) ================")
# Histograms commentary
def span_text(arr, name):
    if not np.isfinite(arr).any():
        return f"{name}: no finite values."
    return f"{name}: span ~ [{np.nanmin(arr):.2f}, {np.nanmax(arr):.2f}], std ~ {np.nanstd(arr):.2f}"

print("• Diversity in inputs (look at histograms you just plotted):")
print("  ", span_text(case_mean_Q, "per-case mean Q"))
print("  ", span_text(case_max_Q,  "per-case max  Q"))
print("  ", span_text(bc_amp,      "BC amplitude sum (Σ|amp|)"))

# Correlation commentary
def corr_text(r, xname):
    if not np.isfinite(r): 
        return f"Correlation({xname}, final T) not available."
    trend = "positive" if r > 0 else "negative"
    mag = abs(r)
    if   mag >= 0.7: lvl = "strong"
    elif mag >= 0.4: lvl = "moderate"
    elif mag >= 0.2: lvl = "weak"
    else:            lvl = "very weak / none"
    return f"{lvl} {trend} correlation (r={r:.2f}) between {xname} and final mean temperature."

print("• Input→Output relationships:")
print("  -", corr_text(r_q,  "per-case mean Q"))
print("  -", corr_text(r_qm, "per-case max Q"))
print("  -", corr_text(r_bc, "BC amplitude sum"))

# ------------------- Final Trainability verdict -------------------
flags = []
# Good PCA if top3 >=85 OR n90<=10
good_pca = (len(cumvar)>=3 and cumvar[2] >= 85.0) or (n90 <= 10)
# Some signal from correlations
some_corr = (abs(r_q) >= 0.2) or (abs(r_qm) >= 0.2) or (abs(r_bc) >= 0.2)
# Diversity check: non-trivial span
diverse_inputs = (np.nanstd(case_mean_Q) > 1e-8) or (np.nanstd(case_max_Q) > 1e-8) or (np.nanstd(bc_amp) > 1e-8)

if good_pca and some_corr and diverse_inputs:
    final_msg = "TRAINABLE: The dataset shows concentrated, structured variation, sensible input→output relationships, and adequate input diversity."
elif good_pca and (some_corr or diverse_inputs):
    final_msg = "LIKELY TRAINABLE: PCA is favorable and at least one of correlation/diversity checks is satisfactory."
elif (some_corr and diverse_inputs):
    final_msg = "BORDERLINE: Inputs are diverse and show some correlation with outputs, but PCA indicates variation is scattered; expect to need more data/regularization."
else:
    final_msg = "NOT IDEAL: Signals are weak across PCA and correlations. Consider increasing diversity of Q/BC sampling or reducing noise."

print("\n================ OVERALL VERDICT ==================")
print(final_msg)


DEEPONET ARCHITECTURE STARTS HERE -->

In [ ]:
import glob
# -----------------------------
# 0) Config / Auto-run directory
# -----------------------------
ROOT = Path(".")
# hardcoded one: RUN_DIR = Path("./dataset_run_YYYYMMDD-HHMMSS")
def _latest_run_dir(root: Path) -> Path | None:
    cands = sorted([Path(p) for p in glob.glob(str(root / "dataset_run_*")) if Path(p).is_dir()])
    return cands[-1] if cands else None

RUN_DIR = _latest_run_dir(ROOT)
assert RUN_DIR is not None, "No dataset_run_* folder found. Build dataset first."

DATA_TEMP = RUN_DIR / "deeponet_temp_dataset.npz"
DATA_SPLIT = RUN_DIR / "splits.json"

print(f"Using RUN_DIR: {RUN_DIR}")

In [ ]:
# -----------------------------
# 1) Load dataset + splits
# -----------------------------
D = np.load(DATA_TEMP, allow_pickle=True)
branch_Q   = D["branch_Q"]            # [C, S_Q_all]
branch_BC  = D["branch_BC"]           # [C, S_BC]
trunk_xy   = D["trunk_xy"]            # [M, 2]
trunk_t    = D["trunk_t"]             # [M, 1]
yT         = D["yT"].reshape(-1, 1)   # [M, 1]
case_ids   = D["case_ids"].astype(np.int64)  # [M]
meta       = D["meta"].item()

N         = int(meta["N"])
Lx, Ly    = float(meta["Lx"]), float(meta["Ly"])
save_times= np.array(meta["save_times"], dtype=float)
S_BC      = int(meta.get("S_BC", branch_BC.shape[1]))

print(f"Loaded: branch_Q{branch_Q.shape}, branch_BC{branch_BC.shape}, "
      f"trunk_xy{trunk_xy.shape}, trunk_t{trunk_t.shape}, yT{yT.shape}, cases={branch_Q.shape[0]}")

if DATA_SPLIT.exists():
    splits = json.loads(Path(DATA_SPLIT).read_text())
    train_ids = np.array(splits["train"], dtype=int)
    val_ids   = np.array(splits["val"], dtype=int)
    test_ids  = np.array(splits["test"], dtype=int)
else:
    C = branch_Q.shape[0]
    idx = np.arange(C); np.random.default_rng(2024).shuffle(idx)
    n_tr = int(round(0.8*C)); n_val = int(round(0.1*C))
    train_ids = idx[:n_tr]; val_ids = idx[n_tr:n_tr+n_val]; test_ids = idx[n_tr+n_val:]
    splits = {"train": train_ids.tolist(), "val": val_ids.tolist(), "test": test_ids.tolist()}
    Path(DATA_SPLIT).write_text(json.dumps(splits, indent=2))
    print("splits.json not found → wrote default 80/10/10 split.")

print(f"Split sizes → train:{len(train_ids)}  val:{len(val_ids)}  test:{len(test_ids)}")


In [ ]:
print("S_Q_all:", branch_Q.shape[1], "| S_BC:", S_BC)
print("Example rows:",
      "bQ:", branch_Q[0,:5],
      "bBC:", branch_BC[0,:5],
      "xy:", trunk_xy[:2],
      "t:", trunk_t[:2].ravel())

In [ ]:
# -----------------------------
# 2) Build samplers / normalization (TRAIN ONLY stats)
# -----------------------------
def _mask_from_cases(allowed_case_ids: np.ndarray) -> np.ndarray:
    allowed = np.zeros(branch_Q.shape[0], dtype=bool)
    allowed[allowed_case_ids] = True
    return allowed[case_ids]  # per-row mask on pooled points

mask_tr = _mask_from_cases(train_ids)
mask_val= _mask_from_cases(val_ids)
mask_te = _mask_from_cases(test_ids)

xy_tr = trunk_xy[mask_tr]       # [M_tr, 2]
t_tr  = trunk_t[mask_tr]        # [M_tr, 1]
y_tr  = yT[mask_tr]
y_mean = float(y_tr.mean())
y_std  = float(y_tr.std() + 1e-6)
cid_tr= case_ids[mask_tr]

xy_val = trunk_xy[mask_val]
t_val  = trunk_t[mask_val]
y_val  = yT[mask_val]
cid_val= case_ids[mask_val]

# branch stats (train cases only) — SEPARATE means/stds for Q and BC
bQ_tr   = branch_Q[train_ids]    # [C_tr, S_Q_all]
bBC_tr  = branch_BC[train_ids]   # [C_tr, S_BC]
bQ_mean = bQ_tr.mean(axis=0, keepdims=True); bQ_std  = bQ_tr.std(axis=0, keepdims=True) + 1e-8
bBC_mean= bBC_tr.mean(axis=0, keepdims=True); bBC_std = bBC_tr.std(axis=0, keepdims=True) + 1e-8

# coordinate/time normalization
xy_min = np.array([0.0, 0.0], dtype=np.float32)
xy_max = np.array([Lx, Ly], dtype=np.float32)
t_min  = float(save_times.min()); t_max = float(save_times.max())

def norm_y(y):
    return ((y - y_mean) / y_std).astype(np.float32)

def denorm_y(y_hat):
    # y_hat is torch tensor; returns torch tensor (Kelvin)
    return y_hat * y_std + y_mean

def norm_branch_Q(bQ):
    return ((bQ - bQ_mean) / bQ_std).astype(np.float32)

def norm_branch_BC(bBC):
    return ((bBC - bBC_mean) / bBC_std).astype(np.float32)

def norm_xy(xy):
    return ((xy - xy_min) / np.maximum(xy_max - xy_min, 1e-6)).astype(np.float32)

def norm_t(tt):
    return ((tt - t_min) / max(t_max - t_min, 1e-6)).astype(np.float32)


In [ ]:
# -----------------------------
# 3) Dataset for pooled point sampling (4-stream)
# -----------------------------
class PooledPointDataset4(Dataset):
    def __init__(self, trunk_xy, trunk_t, yT, case_ids,
                 branch_Q, branch_BC, allowed_case_ids,
                 batch_points=65536):
        self.xy = trunk_xy
        self.tt = trunk_t
        self.y  = yT.astype(np.float32)
        self.case_ids = case_ids
        self.branch_Q = branch_Q
        self.branch_BC= branch_BC

        self.allowed = np.zeros(self.branch_Q.shape[0], dtype=bool)
        self.allowed[allowed_case_ids] = True
        self.rows = np.where(self.allowed[self.case_ids])[0]   # indices eligible
        self.batch_points = int(batch_points)

        # pre-norm trunk
        self.xy_n = norm_xy(self.xy)
        self.tt_n = norm_t(self.tt)

    def __len__(self):
        return 10_000_000  # virtual

    def __getitem__(self, idx):
        ridx = np.random.randint(0, self.rows.shape[0], size=(self.batch_points,))
        rows = self.rows[ridx]

        xy   = self.xy_n[rows]           # [B,2]
        tt   = self.tt_n[rows]           # [B,1]
        y    = self.y[rows]              # [B,1]
        y_n  = norm_y(y)
        cids = self.case_ids[rows]
        bQ   = norm_branch_Q(self.branch_Q[cids])   # [B, S_Q_all]
        bBC  = norm_branch_BC(self.branch_BC[cids]) # [B, S_BC]

        return (torch.from_numpy(bQ),
                torch.from_numpy(bBC),
                torch.from_numpy(xy),
                torch.from_numpy(tt),
                torch.from_numpy(y_n))


In [ ]:
# -----------------------------
# 4) Model: 4-network DeepONet with dot-product head
# -----------------------------
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, act=nn.GELU, dropout=0.1):
        super().__init__()
        layers = []
        dims = (in_dim,) + tuple(hidden) + (out_dim,)
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), act(), nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class DeepONet4(nn.Module):
    """
    y(x,y,t | Q, BC) = ⟨ BQ(Q), BBC(BC), TXY(x,y), TT(t) ⟩_⊙  + b
                      = sum( BQ ⊙ BBC ⊙ TXY ⊙ TT ) + b
    Where each subnetwork outputs a D-dim embedding (shared D).
    """
    def __init__(self, S_Q_all, S_BC,
                 D=256,                       # 256
                 q_hidden=(256,256,256),          #,(256,256,256),
                 bc_hidden=(256,256,256),         #(256,256,256),
                 xy_hidden=(256,256,256,256),     #(256,256,256,256),
                 t_hidden=(128,128)):           #(128,128)
        super().__init__()
        self.branchQ  = MLP(S_Q_all, q_hidden, D)
        self.branchBC = MLP(S_BC,    bc_hidden, D)
        self.trunkXY  = MLP(2,       xy_hidden, D)
        self.trunkT   = MLP(1,       t_hidden,  D)

        # normalizers
        self.lnQ  = nn.LayerNorm(D)
        self.lnBC = nn.LayerNorm(D)
        self.lnXY = nn.LayerNorm(D)
        self.lnT  = nn.LayerNorm(D)

        self.scale4 = (D ** 0.5)

        # NEW: expressive head on concatenation
        self.head = MLP(in_dim=4*D, hidden=(256,128), out_dim=1, act=nn.GELU)
        # tiny learned scalars to weight each path
        self.alpha_concat = nn.Parameter(torch.tensor(alpha_add))
        self.alpha_prod   = nn.Parameter(torch.tensor(alpha_prod))
        self.bias         = nn.Parameter(torch.zeros(1))

    def forward(self, bQ, bBC, xy, tt):
        eQ  = self.lnQ(self.branchQ(bQ))
        eBC = self.lnBC(self.branchBC(bBC))
        eXY = self.lnXY(self.trunkXY(xy))
        eT  = self.lnT(self.trunkT(tt))

        # expressive additive path
        z_concat = torch.cat([eQ, eBC, eXY, eT], dim=1)  # [B, 4D]
        y_concat = self.head(z_concat)                   # [B, 1]

        # multiplicative residual path (scaled)
        y_prod = (eQ * eBC * eXY * eT).sum(dim=1, keepdim=True) / self.scale4

        # combine
        yhat_norm = self.alpha_concat * y_concat + self.alpha_prod * y_prod + self.bias
        return yhat_norm

In [ ]:
import torch, importlib
print(torch.__version__)
importlib.import_module("torch._utils")
import torch.optim as optim
optim.Adam([torch.nn.Parameter(torch.randn(2,requires_grad=True))], lr=1e-3)
print("Adam OK")

In [ ]:
# -----------------------------
# 5) Train / Validate
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")
print("\n")

S_Q_all = branch_Q.shape[1]
model = DeepONet4(S_Q_all=S_Q_all, S_BC=S_BC,
                  D=128,
                  q_hidden=(128,128),
                  bc_hidden=(128,128),
                  xy_hidden=(128,128,128),
                  t_hidden=(64,64)).to(device)


BATCH_POINTS = BATCH_POINTS
EPOCHS = EPOCHS
LR = LR
WEIGHT_DECAY = WEIGHT_DECAY
VAL_SAMPLES = VAL_SAMPLES
VAL_BATCH_POINTS = VAL_BATCH_POINTS
STEPS_PER_EPOCH = STEPS_PER_EPOCH


ds_tr = PooledPointDataset4(trunk_xy=xy_tr, trunk_t=t_tr, yT=y_tr, case_ids=cid_tr,
                            branch_Q=branch_Q, branch_BC=branch_BC, allowed_case_ids=train_ids,
                            batch_points=BATCH_POINTS)
ds_va = PooledPointDataset4(trunk_xy=xy_val, trunk_t=t_val, yT=y_val, case_ids=cid_val,
                            branch_Q=branch_Q, branch_BC=branch_BC, allowed_case_ids=val_ids,
                            batch_points=VAL_BATCH_POINTS)
ds_tr.batch_points = BATCH_POINTS
ds_va.batch_points = BATCH_POINTS

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)  # safer defaults
best_val = math.inf
ckpt_path = RUN_DIR / "deeponet_temp_model_4net.pt"

print("Starting training…")
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss_acc = 0.0
    train_rmseK_acc = 0.0

    for _ in range(STEPS_PER_EPOCH):
        bQ, bBC, xy, tt, ytrue_n = ds_tr[0]
        bQ=bQ.to(device); bBC=bBC.to(device); xy=xy.to(device); tt=tt.to(device); ytrue_n=ytrue_n.to(device)

        opt.zero_grad(set_to_none=True)
        yhat_n = model(bQ, bBC, xy, tt)
        loss = ((yhat_n - ytrue_n)**2).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        with torch.no_grad():
            # report RMSE(K) on this mini-batch
            yhat_K = denorm_y(yhat_n)
            y_K    = denorm_y(ytrue_n)
            train_rmseK_acc += torch.sqrt(((yhat_K - y_K)**2).mean()).item()
            train_loss_acc  += loss.item()

    # validation over a few random batches
    model.eval()
    with torch.no_grad():
        vloss_acc = 0.0
        rmseK_acc = 0.0
        for _ in range(VAL_SAMPLES):
            bQv, bBCv, xyv, ttv, yv_n = ds_va[0]
            bQv=bQv.to(device); bBCv=bBCv.to(device); xyv=xyv.to(device); ttv=ttv.to(device); yv_n=yv_n.to(device)
            yhatv_n = model(bQv, bBCv, xyv, ttv)
            vloss_acc += ((yhatv_n - yv_n)**2).mean().item()
            yhatv_K = denorm_y(yhatv_n); yv_K = denorm_y(yv_n)
            rmseK_acc += torch.sqrt(((yhatv_K - yv_K)**2).mean()).item()

    train_mse_n  = train_loss_acc / STEPS_PER_EPOCH
    train_rmse_K = train_rmseK_acc / STEPS_PER_EPOCH
    vloss        = vloss_acc / VAL_SAMPLES
    val_rmseK    = rmseK_acc / VAL_SAMPLES

    print(f"alphas → concat={model.alpha_concat.item():.4f} | prod={model.alpha_prod.item():.4f}")

    print(f"Epoch {epoch:03d} | train MSE(n){train_mse_n:.6e} | val MSE(n){vloss:.6e} "
          f"| train RMSE(K){train_rmse_K:.3f} | val RMSE(K){val_rmseK:.3f}")

    if vloss < best_val:
        best_val = vloss
        torch.save({
            "model": model.state_dict(),
            "bQ_mean": bQ_mean, "bQ_std": bQ_std,
            "bBC_mean": bBC_mean, "bBC_std": bBC_std,
            "xy_min": xy_min, "xy_max": xy_max,
            "t_min": t_min, "t_max": t_max,
            "y_mean": y_mean, "y_std": y_std,
            "S_Q_all": branch_Q.shape[1], "S_BC": S_BC,
            "meta": meta,
        }, ckpt_path)

print(f"Best val MSE(n): {best_val:.6e}")
print(f"Saved checkpoint → {ckpt_path}")


In [ ]:
#-------------------------------
# 6) INFERENCE PLOTS
#-------------------------------

@torch.no_grad()
def load_trained(path: Path):
    ck = torch.load(path, map_location="cpu",weights_only=False)
    # Try to read minimal shape config; fall back to defaults if missing
    S_Q_all = ck.get("S_Q_all", ck.get("S_Q", None))
    S_BC    = ck.get("S_BC", None)
    assert S_Q_all is not None and S_BC is not None, "Checkpoint missing S_Q_all/S_BC"

    # Instantiate your DeepONet4 with defaults used at train time
    m = DeepONet4(S_Q_all=S_Q_all, S_BC=S_BC).to(device)
    m.load_state_dict(ck["model"], strict=True)
    m.eval()

    # Stats for normalization / denormalization
    stats = {
        "bQ_mean": ck["bQ_mean"], "bQ_std": ck["bQ_std"],
        "bBC_mean": ck["bBC_mean"], "bBC_std": ck["bBC_std"],
        "xy_min": ck["xy_min"], "xy_max": ck["xy_max"],
        "t_min": ck["t_min"], "t_max": ck["t_max"],
        "y_mean": ck.get("y_mean", 0.0), "y_std": ck.get("y_std", 1.0),
        "S_Q_all": S_Q_all, "S_BC": S_BC,
        "meta": ck["meta"],
    }
    return m, stats

def _norm_branch_separate(stats, bQ_case, bBC_case):
    bQn  = (bQ_case[None, :]  - stats["bQ_mean"])  / (stats["bQ_std"]  + 1e-8)  # [1, S_Q_all]
    bBCn = (bBC_case[None, :] - stats["bBC_mean"]) / (stats["bBC_std"] + 1e-8)  # [1, S_BC]
    return bQn.astype(np.float32), bBCn.astype(np.float32)

def _norm_xy_for(stats, xy):
    return ((xy - stats["xy_min"]) / np.maximum(stats["xy_max"] - stats["xy_min"], 1e-6)).astype(np.float32)

def _norm_t_for(stats, tt):
    return ((tt - stats["t_min"]) / max(stats["t_max"] - stats["t_min"], 1e-6)).astype(np.float32)

def _denorm_y_for(stats, y_norm_tensor):
    # tensor -> Kelvin (tensor)
    return y_norm_tensor * stats["y_std"] + stats["y_mean"]

def _upsample_nn(coarse: np.ndarray, N: int) -> np.ndarray:
    """Nearest-neighbor upsample from SxS -> NxN (no external deps)."""
    S = coarse.shape[0]
    if S == N:
        return coarse.astype(np.float32)
    xi = (np.linspace(0, S-1, N)).round().astype(int)
    yi = (np.linspace(0, S-1, N)).round().astype(int)
    return coarse[np.ix_(yi, xi)].astype(np.float32)

def _reconstruct_Q_map_from_branchQ(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Our branch_Q is concatenated across Nt snapshots.
    For static Q, any one block (per-time length) is fine. Use the FIRST block.
    """
    N   = int(meta_local["N"])
    Nt  = len(meta_local["save_times"])
    mode = meta_local.get("sensor_mode", "full")
    bQ_all = branch_Q[case_id]  # [S_Q_all]

    if mode == "full":
        per_time = N * N
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length N*N."
        return bQ_all[:per_time].reshape(N, N).astype(np.float32)
    else:
        S = int(meta_local.get("S_down", 40))
        per_time = S * S
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length S_down*S_down."
        coarse = bQ_all[:per_time].reshape(S, S)
        return _upsample_nn(coarse, N)

def _reconstruct_true_T_maps(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Rebuild true T maps from branch_T (if available) or skip if not needed.
    We use the frac dataset's branch_T which stores concatenated snapshot grids.
    """
    N = int(meta_local["N"])
    times = np.array(meta_local["save_times"], dtype=float)
    Nt = len(times)
    mode = meta_local.get("sensor_mode", "full")

    Dfrac = np.load(RUN_DIR / "deeponet_frac_dataset.npz", allow_pickle=True)
    Tmat = Dfrac["branch_T"][case_id]  # [S_T = per_time * Nt]

    if mode == "full":
        return Tmat.reshape(Nt, N, N).astype(np.float32)
    else:
        S = int(meta_local.get("S_down", 40))
        coarse = Tmat.reshape(Nt, S, S)
        return np.stack([_upsample_nn(coarse[k], N) for k in range(Nt)], axis=0)

@torch.no_grad()
def predict_case_maps(ckpt: Path, case_id: int, sel_times: np.ndarray):
    model, stats = load_trained(ckpt)
    meta_local = stats["meta"]
    N   = int(meta_local["N"]); Lx = float(meta_local["Lx"]); Ly = float(meta_local["Ly"])

    # branch vecs (separate normalization)
    bQ_case  = branch_Q[case_id]    # [S_Q_all]
    bBC_case = branch_BC[case_id]   # [S_BC]
    bQn, bBCn = _norm_branch_separate(stats, bQ_case, bBC_case)  # each [1, *]
    bQ_t  = torch.from_numpy(bQn).to(device)   # [1, S_Q_all]
    bBC_t = torch.from_numpy(bBCn).to(device)  # [1, S_BC]

    # grid (N x N)
    xs = np.linspace(0.0, Lx, N, dtype=np.float32)
    ys = np.linspace(0.0, Ly, N, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    XY = np.stack([X, Y], axis=-1).reshape(-1, 2)  # [N*N, 2]
    XY_n = _norm_xy_for(stats, XY)                 # [N*N, 2]
    XY_t = torch.from_numpy(XY_n).to(device)

    outs = []
    for t in sel_times:
        tt = np.full((XY.shape[0], 1), float(t), dtype=np.float32)  # [N*N, 1]
        tt_n = _norm_t_for(stats, tt)                               # [N*N, 1]
        tt_t = torch.from_numpy(tt_n).to(device)

        # tile branch to match grid points
        B = XY_t.shape[0]
        bQ_tile  = bQ_t.repeat(B, 1)    # [B, S_Q_all]
        bBC_tile = bBC_t.repeat(B, 1)   # [B, S_BC]

        # model outputs normalized T → denormalize
        yhat_n = model(bQ_tile, bBC_tile, XY_t, tt_t)             # [B,1] normalized
        yhat_K = _denorm_y_for(stats, yhat_n).cpu().numpy()       # [B,1] Kelvin
        outs.append(yhat_K.reshape(N, N).astype(np.float32))
    return np.stack(outs, axis=0)  # [Tsel, N, N]

def _pick_times(times_all: np.ndarray, num_cols: int = 5, prefer: np.ndarray | None = None):
    """Pick ~evenly spaced time stamps (or nearest to 'prefer' if provided)."""
    times_all = np.array(times_all, dtype=float)
    if prefer is None:
        if len(times_all) <= num_cols:
            return times_all
        idx = np.linspace(0, len(times_all)-1, num_cols).round().astype(int)
        return times_all[idx]
    # map preferred to nearest in times_all
    out = []
    for t in prefer:
        out.append(times_all[np.argmin(np.abs(times_all - t))])
    # keep unique in order
    uniq = []
    for t in out:
        if len(uniq)==0 or abs(uniq[-1]-t) > 1e-12:
            uniq.append(t)
    return np.array(uniq[:num_cols], dtype=float)

def plot_case_transient(ckpt_path: Path, case_id: int, num_cols: int = 5, prefer_times=None):
    # meta from dataset file we loaded earlier (global 'meta')
    meta_local = meta
    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"]); Ly = float(meta_local["Ly"])
    times_all = np.array(meta_local["save_times"], dtype=float)
    sel_times = _pick_times(times_all, num_cols=num_cols, prefer=prefer_times)

    # data: Q map, true T maps, predicted T maps
    Q_map      = _reconstruct_Q_map_from_branchQ(case_id, meta_local)   # [N,N]
    T_true_all = _reconstruct_true_T_maps(case_id, meta_local)          # [Nt,N,N]
    idx_true   = np.array([np.argmin(np.abs(times_all - t)) for t in sel_times], dtype=int)
    T_true     = T_true_all[idx_true]                                   # [Tsel,N,N]
    T_pred     = predict_case_maps(ckpt_path, case_id, sel_times)       # [Tsel,N,N]

    # consistent color scaling for T
    vmin = min(T_true.min(), T_pred.min())
    vmax = max(T_true.max(), T_pred.max())

    cols = len(sel_times)
    fig, axes = plt.subplots(nrows=3, ncols=cols, figsize=(3.2*cols, 9.0), constrained_layout=True)

    # Row 0: Heat source (repeat same Q for alignment)
    for c in range(cols):
        ax = axes[0, c]
        im = ax.imshow(Q_map.T, origin="lower", extent=[0, Lx, 0, Ly], cmap="RdBu_r")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0: ax.set_ylabel("Heat source Q", fontsize=11)
        ax.set_title(f"t = {sel_times[c]:g}s", fontsize=11)
    fig.colorbar(im, ax=axes[0, :].ravel().tolist(), fraction=0.02, pad=0.02)

    # Row 1: True T
    for c in range(cols):
        ax = axes[1, c]
        imT = ax.imshow(T_true[c].T, origin="lower", extent=[0, Lx, 0, Ly],cmap="inferno", vmin=vmin, vmax=vmax,aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0: ax.set_ylabel("True T", fontsize=11)
    fig.colorbar(imT, ax=axes[1, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("T[K]")

    # Row 2: Pred T
    for c in range(cols):
        ax = axes[2, c]
        imP = ax.imshow(T_pred[c].T, origin="lower", extent=[0, Lx, 0, Ly], cmap="inferno",vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0: ax.set_ylabel("Pred T", fontsize=11)
    fig.colorbar(imP, ax=axes[2, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("T[K]")

    plt.show()


In [ ]:
# Pick a demo case and plot
ckpt_path = Path(r'runs/dataset_run_20251120-131248/deeponet_temp_model_4net.pt')

for i in range(len(test_ids)):
    demo_case = int(test_ids[i]) # if len(test_ids) else val_ids[i])
    plot_case_transient(ckpt_path, demo_case, num_cols=7,prefer_times=save_times)
    print(demo_case+1) # 0-based indexing

### LIQUID FRACTION PREDICTIONS

In [ ]:
import glob
# -----------------------------
# 0) Config / Auto-run directory
# -----------------------------
ROOT = Path(".")
# hardcoded one: RUN_DIR = Path("./dataset_run_YYYYMMDD-HHMMSS")
def _latest_run_dir(root: Path) -> Path | None:
    cands = sorted([Path(p) for p in glob.glob(str(root / "dataset_run_*")) if Path(p).is_dir()])
    return cands[-1] if cands else None

RUN_DIR = _latest_run_dir(ROOT)
assert RUN_DIR is not None, "No dataset_run_* folder found. Build dataset first."

DATA_FRAC = RUN_DIR / "deeponet_frac_dataset.npz"
DATA_TEMP = RUN_DIR / "deeponet_temp_dataset.npz"
DATA_SPLIT = RUN_DIR / "splits.json"

print(f"Using RUN_DIR: {RUN_DIR}")

In [ ]:
# -----------------------------
# 1) Load dataset + splits
# -----------------------------
D = np.load(DATA_FRAC, allow_pickle=True)
D2 = np.load(DATA_TEMP, allow_pickle=True)
# ---- per-case branch features ----
branch_T   = D["branch_T"]                  # [C, S_Tsnap]  temperature snapshots for frac
branch_Q   = D2["branch_Q"]                 # [C, S_Q_all]  heat-source field Q from temp dataset
branch_BC  = D2["branch_BC"]                # [C, S_BC]     boundary conditions from temp dataset

# sanity check: same number of cases in both datasets
C_frac = branch_T.shape[0]
C_temp_Q  = branch_Q.shape[0]
C_temp_BC = branch_BC.shape[0]
assert C_frac == C_temp_Q == C_temp_BC, \
    f"Case count mismatch: frac={C_frac}, temp_Q={C_temp_Q}, temp_BC={C_temp_BC}"

# ---- point-level trunk + targets from frac dataset ----
trunk_xy   = D["trunk_xy"]                  # [M, 2]
trunk_t    = D["trunk_t"]                   # [M, 1]
yf         = D["yf"].reshape(-1, 1)         # [M, 1] liquid fraction
case_ids   = D["case_ids"].astype(np.int64) # [M]   (case index for each point)
meta       = D["meta"].item()

N          = int(meta["N"])
Lx, Ly     = float(meta["Lx"]), float(meta["Ly"])
save_times = np.array(meta["save_times"], dtype=float)

print(f"Loaded FRAC : branch_T{branch_T.shape}, "
      f"trunk_xy{trunk_xy.shape}, trunk_t{trunk_t.shape}, yf{yf.shape}")
print(f"Loaded TEMP : branch_Q{branch_Q.shape}, branch_BC{branch_BC.shape}")

# ---- case-level splits (same style as before) ----
DATA_SPLIT = Path(DATA_SPLIT)  # ensure Path
if DATA_SPLIT.exists():
    splits = json.loads(Path(DATA_SPLIT).read_text())
    train_ids = np.array(splits["train"], dtype=int)
    val_ids   = np.array(splits["val"],   dtype=int)
    test_ids  = np.array(splits["test"],  dtype=int)
else:
    C = branch_T.shape[0]
    idx = np.arange(C)
    np.random.default_rng(2024).shuffle(idx)
    n_tr  = int(round(0.8 * C))
    n_val = int(round(0.1 * C))
    train_ids = idx[:n_tr]
    val_ids   = idx[n_tr:n_tr + n_val]
    test_ids  = idx[n_tr + n_val:]
    splits = {
        "train": train_ids.tolist(),
        "val":   val_ids.tolist(),
        "test":  test_ids.tolist()
    }
    Path(DATA_SPLIT).write_text(json.dumps(splits, indent=2))
    print("splits.json not found → wrote default 80/10/10 split.")

print(f"Split sizes → train:{len(train_ids)}  val:{len(val_ids)}  test:{len(test_ids)}")


In [ ]:
#checker
print("S_T_all:", branch_T.shape[1])
print("Example rows:",
      "bT:", branch_T[0,:5],
      "xy:", trunk_xy[:2],
      "t:", trunk_t[:2].ravel())

In [ ]:
def _mask_from_cases(allowed_case_ids: np.ndarray) -> np.ndarray:
    allowed = np.zeros(branch_T.shape[0], dtype=bool)
    allowed[allowed_case_ids] = True
    return allowed[case_ids]


mask_tr = _mask_from_cases(train_ids)
mask_val = _mask_from_cases(val_ids)
mask_te  = _mask_from_cases(test_ids)

# -------------------------
# point-level tensors
# -------------------------
xy_tr = trunk_xy[mask_tr]        # [M_tr, 2]
t_tr  = trunk_t[mask_tr]         # [M_tr, 1]
y_tr  = yf[mask_tr]              # [M_tr, 1]
y_mean = float(y_tr.mean())
y_std  = float(y_tr.std() + 1e-6)
cid_tr = case_ids[mask_tr]

xy_val = trunk_xy[mask_val]
t_val  = trunk_t[mask_val]
y_val  = yf[mask_val]
cid_val = case_ids[mask_val]

# ============================================================
# branch-level stats (TRAIN CASES ONLY)
# ============================================================

# ----- Temperature T-snapshot branch -----
bT_tr   = branch_T[train_ids]               # [C_tr, S_Tsnap]
bT_mean = bT_tr.mean(axis=0, keepdims=True)
bT_std  = bT_tr.std(axis=0, keepdims=True) + 1e-8

# ----- Q branch -----
bQ_tr   = branch_Q[train_ids]               # [C_tr, S_Q_all]
bQ_mean = bQ_tr.mean(axis=0, keepdims=True)
bQ_std  = bQ_tr.std(axis=0, keepdims=True) + 1e-8

# ----- BC branch -----
bBC_tr   = branch_BC[train_ids]             # [C_tr, S_BC]
bBC_mean = bBC_tr.mean(axis=0, keepdims=True)
bBC_std  = bBC_tr.std(axis=0, keepdims=True) + 1e-8

# ============================================================
# coordinate + time normalization
# ============================================================
xy_min = np.array([0.0, 0.0], dtype=np.float32)
xy_max = np.array([Lx, Ly], dtype=np.float32)
t_min  = float(save_times.min())
t_max  = float(save_times.max())


# ============================================================
# Normalization functions
# ============================================================

def norm_y(y):
    return ((y - y_mean) / y_std).astype(np.float32)

def denorm_y(y_hat):
    # y_hat is torch tensor
    return y_hat * y_std + y_mean


def norm_branch_T(bT):
    return ((bT - bT_mean) / bT_std).astype(np.float32)

def norm_branch_Q(bQ):
    return ((bQ - bQ_mean) / bQ_std).astype(np.float32)

def norm_branch_BC(bBC):
    return ((bBC - bBC_mean) / bBC_std).astype(np.float32)


def norm_xy(xy):
    return ((xy - xy_min) / np.maximum(xy_max - xy_min, 1e-6)).astype(np.float32)

def norm_t(tt):
    return ((tt - t_min) / max(t_max - t_min, 1e-6)).astype(np.float32)


In [ ]:
# -----------------------------
# 3) Dataset for pooled point sampling (5-stream: Q, BC, T, xy, t)
# -----------------------------
class PooledPointDataset5(Dataset):
    def __init__(self, trunk_xy, trunk_t, yf, case_ids,
                 branch_Q, branch_BC, branch_T, allowed_case_ids,
                 batch_points=65536):
        self.xy       = trunk_xy
        self.tt       = trunk_t
        self.y        = yf.astype(np.float32)
        self.case_ids = case_ids

        # per-case branch data
        self.branch_Q  = branch_Q
        self.branch_BC = branch_BC
        self.branch_T  = branch_T

        # which cases are allowed in this split (train/val/test)
        self.allowed = np.zeros(self.branch_T.shape[0], dtype=bool)
        self.allowed[allowed_case_ids] = True

        # rows (point indices) whose case_ids are in allowed_case_ids
        self.rows = np.where(self.allowed[self.case_ids])[0]   # indices eligible

        self.batch_points = int(batch_points)

        # pre-normalize trunk
        self.xy_n = norm_xy(self.xy)   # [M, 2]
        self.tt_n = norm_t(self.tt)    # [M, 1]

    def __len__(self):
        return 10_000_000  # virtual

    def __getitem__(self, idx):
        # sample random point indices from the eligible rows
        ridx = np.random.randint(0, self.rows.shape[0], size=(self.batch_points,))
        rows = self.rows[ridx]

        xy   = self.xy_n[rows]           # [B, 2]
        tt   = self.tt_n[rows]           # [B, 1]
        y    = self.y[rows]              # [B, 1]
        y_n  = norm_y(y)                 # normalized target

        cids = self.case_ids[rows]       # [B]

        # normalize branch features per point (by case)
        bQ   = norm_branch_Q(self.branch_Q[cids])    # [B, S_Q_all]
        bBC  = norm_branch_BC(self.branch_BC[cids])  # [B, S_BC]
        bT   = norm_branch_T(self.branch_T[cids])    # [B, S_Tsnap]

        return (torch.from_numpy(bQ),
                torch.from_numpy(bBC),
                torch.from_numpy(bT),
                torch.from_numpy(xy),
                torch.from_numpy(tt),
                torch.from_numpy(y_n))


In [ ]:
# -----------------------------
# 4) Model: 5-network DeepONet with dot-product head
#      Branch: Q, BC, T
#      Trunk : (x,y), t
# -----------------------------
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, act=nn.GELU, dropout=0.1):
        super().__init__()
        layers = []
        dims = (in_dim,) + tuple(hidden) + (out_dim,)
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), act(), nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)
    def forward(self, x): 
        return self.net(x)


class DeepONet5(nn.Module):
    """
    y(x,y,t | Q, BC, T) = α_concat * f_concat(Q,BC,T,xy,t)
                        + α_prod   * f_prod(Q,BC,T,xy,t)
                        + b

    where:
      - 5 subnetworks each output a D-dim embedding:
            branchQ  : Q-field features
            branchBC : boundary-condition features
            branchT  : temperature-snapshot features
            trunkXY  : spatial (x,y)
            trunkT   : time t
      - f_concat uses concatenation of all 5 embeddings and an expressive MLP head.
      - f_prod   uses elementwise product of all 5 embeddings, summed and scaled.
    """
    def __init__(self,
                 S_Q_all,          # length of Q feature vector
                 S_BC,             # length of BC feature vector
                 S_T_all,          # length of T-snapshot feature vector
                 D=256,
                 Q_hidden=(256,256,256),
                 BC_hidden=None,
                 T_hidden=(256,256,256),
                 xy_hidden=(256,256,256,256),
                 t_hidden=(128,128)):
        super().__init__()

        if BC_hidden is None:
            BC_hidden = Q_hidden

        # ---------------- Branch networks ----------------
        self.branchQ  = MLP(S_Q_all, Q_hidden,  D)
        self.branchBC = MLP(S_BC,    BC_hidden, D)
        self.branchT  = MLP(S_T_all, T_hidden,  D)

        # ---------------- Trunk networks -----------------
        self.trunkXY  = MLP(2,       xy_hidden, D)
        self.trunkT   = MLP(1,       t_hidden,  D)

        # ---------------- LayerNorms ---------------------
        self.lnQ   = nn.LayerNorm(D)
        self.lnBC  = nn.LayerNorm(D)
        self.lnT   = nn.LayerNorm(D)
        self.lnXY  = nn.LayerNorm(D)
        self.lnTau = nn.LayerNorm(D)

        # scaling for multiplicative path
        self.scale5 = (D ** 0.5)

        # ---------------- Expressive head ----------------
        # concatenation of 5 embeddings → 5D
        self.head = MLP(in_dim=5*D, hidden=(256,128), out_dim=1, act=nn.GELU)

        # tiny learned scalars to weight each path
        # (assumes alpha_add and alpha_prod are defined above, as in your original code)
        self.alpha_concat = nn.Parameter(torch.tensor(alpha_add))
        self.alpha_prod   = nn.Parameter(torch.tensor(alpha_prod))
        self.bias         = nn.Parameter(torch.zeros(1))

    def forward(self, bQ, bBC, bT, xy, tt):
        """
        bQ  : [B, S_Q_all]
        bBC : [B, S_BC]
        bT  : [B, S_T_all]
        xy  : [B, 2]
        tt  : [B, 1]
        """
        # embeddings + layer norms
        eQ   = self.lnQ(self.branchQ(bQ))       # [B, D]
        eBC  = self.lnBC(self.branchBC(bBC))    # [B, D]
        eT   = self.lnT(self.branchT(bT))       # [B, D]
        eXY  = self.lnXY(self.trunkXY(xy))      # [B, D]
        eTau = self.lnTau(self.trunkT(tt))      # [B, D]

        # ----- expressive additive path (concat) -----
        z_concat = torch.cat([eQ, eBC, eT, eXY, eTau], dim=1)  # [B, 5D]
        y_concat = self.head(z_concat)                         # [B, 1]

        # ----- multiplicative residual path -----
        e_prod = eQ * eBC * eT * eXY * eTau                    # [B, D]
        y_prod = e_prod.sum(dim=1, keepdim=True) / self.scale5 # [B, 1]

        # ----- combine -----
        yhat_norm = self.alpha_concat * y_concat + self.alpha_prod * y_prod + self.bias
        return yhat_norm


In [ ]:
import torch, importlib
print(torch.__version__)
importlib.import_module("torch._utils")
import torch.optim as optim
optim.Adam([torch.nn.Parameter(torch.randn(2,requires_grad=True))], lr=1e-3)
print("Adam OK")

In [ ]:
# -----------------------------
# 5) Train / Validate
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")
print("\n")

S_Q_all  = branch_Q.shape[1]
S_BC     = branch_BC.shape[1]
S_T_all  = branch_T.shape[1]

model = DeepONet5(S_Q_all=S_Q_all,
                  S_BC=S_BC,
                  S_T_all=S_T_all,
                  D=256,
                  Q_hidden=(256,256,256),
                  BC_hidden=None,
                  T_hidden=(256,256,256),
                  xy_hidden=(256,256,256,256),
                  t_hidden=(128,128)).to(device)

BATCH_POINTS     = BATCH_POINTS
VAL_BATCH_POINTS = VAL_BATCH_POINTS
EPOCHS           = EPOCHS
LR               = LR
WEIGHT_DECAY     = WEIGHT_DECAY
VAL_SAMPLES      = VAL_SAMPLES
STEPS_PER_EPOCH  = STEPS_PER_EPOCH

# note: we use the full arrays trunk_xy, trunk_t, yf, case_ids
# and let allowed_case_ids control which cases are sampled
ds_tr = PooledPointDataset5(trunk_xy=trunk_xy, trunk_t=trunk_t, yf=yf, case_ids=case_ids,
                            branch_Q=branch_Q, branch_BC=branch_BC, branch_T=branch_T,
                            allowed_case_ids=train_ids,
                            batch_points=BATCH_POINTS)

ds_va = PooledPointDataset5(trunk_xy=trunk_xy, trunk_t=trunk_t, yf=yf, case_ids=case_ids,
                            branch_Q=branch_Q, branch_BC=branch_BC, branch_T=branch_T,
                            allowed_case_ids=val_ids,
                            batch_points=VAL_BATCH_POINTS)

ds_tr.batch_points = BATCH_POINTS
ds_va.batch_points = VAL_BATCH_POINTS

opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_val = math.inf
ckpt_path = RUN_DIR / "deeponet_frac_model_5net.pt"

print("Starting training…")
for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss_acc = 0.0
    train_rmse_acc = 0.0

    for _ in range(STEPS_PER_EPOCH):
        bQ, bBC, bT, xy, tt, ytrue_n = ds_tr[0]
        bQ = bQ.to(device)
        bBC = bBC.to(device)
        bT = bT.to(device)
        xy = xy.to(device)
        tt = tt.to(device)
        ytrue_n = ytrue_n.to(device)

        opt.zero_grad(set_to_none=True)
        yhat_n = model(bQ, bBC, bT, xy, tt)
        loss = ((yhat_n - ytrue_n)**2).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        with torch.no_grad():
            # RMSE in physical space (here: liquid fraction units)
            yhat_phys = denorm_y(yhat_n)
            y_phys    = denorm_y(ytrue_n)
            train_rmse_acc += torch.sqrt(((yhat_phys - y_phys)**2).mean()).item()
            train_loss_acc += loss.item()

    # ----------------- validation -----------------
    model.eval()
    with torch.no_grad():
        vloss_acc = 0.0
        rmse_acc  = 0.0
        for _ in range(VAL_SAMPLES):
            bQv, bBCv, bTv, xyv, ttv, yv_n = ds_va[0]
            bQv = bQv.to(device)
            bBCv = bBCv.to(device)
            bTv = bTv.to(device)
            xyv = xyv.to(device)
            ttv = ttv.to(device)
            yv_n = yv_n.to(device)

            yhatv_n = model(bQv, bBCv, bTv, xyv, ttv)
            vloss_acc += ((yhatv_n - yv_n)**2).mean().item()

            yhatv_phys = denorm_y(yhatv_n)
            yv_phys    = denorm_y(yv_n)
            rmse_acc += torch.sqrt(((yhatv_phys - yv_phys)**2).mean()).item()

    train_mse_n  = train_loss_acc / STEPS_PER_EPOCH
    train_rmse   = train_rmse_acc / STEPS_PER_EPOCH
    vloss        = vloss_acc / VAL_SAMPLES
    val_rmse     = rmse_acc / VAL_SAMPLES

    print(f"alphas → concat={model.alpha_concat.item():.4f} | prod={model.alpha_prod.item():.4f}")
    print(f"Epoch {epoch:03d} | train MSE(n){train_mse_n:.6e} | val MSE(n){vloss:.6e} "
          f"| train RMSE{train_rmse:.3f} | val RMSE{val_rmse:.3f}")

    if vloss < best_val:
        best_val = vloss
        torch.save({
            "model": model.state_dict(),
            "bQ_mean": bQ_mean,  "bQ_std":  bQ_std,
            "bBC_mean": bBC_mean, "bBC_std": bBC_std,
            "bT_mean": bT_mean,  "bT_std":  bT_std,
            "xy_min": xy_min,    "xy_max":  xy_max,
            "t_min": t_min,      "t_max":   t_max,
            "y_mean": y_mean,    "y_std":   y_std,
            "S_Q_all": S_Q_all,
            "S_BC": S_BC,
            "S_T_all": S_T_all,
            "meta": meta,
        }, ckpt_path)
        print("  -> checkpoint saved.")

print(f"Best val MSE(n): {best_val:.6e}")
print(f"Saved checkpoint → {ckpt_path}")


In [ ]:
#-------------------------------
# 6) INFERENCE PLOTS (liq fraction)
#-------------------------------

@torch.no_grad()
def load_trained(path: Path):
    ck = torch.load(path, map_location="cpu", weights_only=False)

    # Shapes from checkpoint
    S_Q_all = ck.get("S_Q_all", None)
    S_BC    = ck.get("S_BC", None)
    S_T_all = ck.get("S_T_all", None)
    assert S_Q_all is not None and S_BC is not None and S_T_all is not None, \
        "Checkpoint missing S_Q_all / S_BC / S_T_all"

    # Instantiate your DeepONet3 with same defaults used at train time
    m = DeepONet5(S_Q_all=S_Q_all,
                  S_BC=S_BC,
                  S_T_all=S_T_all,
                  D=256,
                  Q_hidden=(256,256,256),
                  BC_hidden=None,
                  T_hidden=(256,256,256),
                  xy_hidden=(256,256,256,256),
                  t_hidden=(128,128)).to(device)
    m.load_state_dict(ck["model"], strict=True)
    m.eval()

    # Stats for normalization / denormalization
    stats = {
        "bQ_mean": ck["bQ_mean"],  "bQ_std":  ck["bQ_std"],
        "bBC_mean": ck["bBC_mean"], "bBC_std": ck["bBC_std"],
        "bT_mean": ck["bT_mean"],  "bT_std":  ck["bT_std"],
        "xy_min": ck["xy_min"],    "xy_max":  ck["xy_max"],
        "t_min": ck["t_min"],      "t_max":   ck["t_max"],
        "y_mean": ck.get("y_mean", 0.0), 
        "y_std":  ck.get("y_std",  1.0),
        "S_Q_all": S_Q_all, 
        "S_BC":    S_BC,
        "S_T_all": S_T_all,
        "meta":    ck["meta"],
    }
    return m, stats


def _norm_branch_for(stats, bQ_case, bBC_case, bT_case):
    bQn  = (bQ_case[None, :]  - stats["bQ_mean"])  / (stats["bQ_std"]  + 1e-8)  # [1, S_Q_all]
    bBCn = (bBC_case[None, :] - stats["bBC_mean"]) / (stats["bBC_std"] + 1e-8)  # [1, S_BC]
    bTn  = (bT_case[None, :]  - stats["bT_mean"])  / (stats["bT_std"]  + 1e-8)  # [1, S_T_all]
    return (bQn.astype(np.float32),
            bBCn.astype(np.float32),
            bTn.astype(np.float32))


def _norm_xy_for(stats, xy):
    return ((xy - stats["xy_min"]) / np.maximum(stats["xy_max"] - stats["xy_min"], 1e-6)).astype(np.float32)


def _norm_t_for(stats, tt):
    return ((tt - stats["t_min"]) / max(stats["t_max"] - stats["t_min"], 1e-6)).astype(np.float32)


def _denorm_y_for(stats, y_norm_tensor):
    # tensor -> physical fraction (tensor)
    return y_norm_tensor * stats["y_std"] + stats["y_mean"]


def _upsample_nn(coarse: np.ndarray, N: int) -> np.ndarray:
    """Nearest-neighbor upsample from SxS -> NxN (no external deps)."""
    S = coarse.shape[0]
    if S == N:
        return coarse.astype(np.float32)
    xi = (np.linspace(0, S-1, N)).round().astype(int)
    yi = (np.linspace(0, S-1, N)).round().astype(int)
    return coarse[np.ix_(yi, xi)].astype(np.float32)


def _reconstruct_Q_map_from_branchQ(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Our branch_Q is concatenated across Nt snapshots.
    For static Q, any one block (per-time length) is fine. Use the FIRST block.
    """
    N   = int(meta_local["N"])
    Nt  = len(meta_local["save_times"])
    mode = meta_local.get("sensor_mode", "full")
    bQ_all = branch_Q[case_id]  # [S_Q_all]

    if mode == "full":
        per_time = N * N
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length N*N."
        return bQ_all[:per_time].reshape(N, N).astype(np.float32)
    else:
        S = int(meta_local.get("S_down", 40))
        per_time = S * S
        assert bQ_all.shape[0] % Nt == 0 and (bQ_all.shape[0] // Nt) == per_time, \
            "branch_Q does not match expected per-time length S_down*S_down."
        coarse = bQ_all[:per_time].reshape(S, S)
        return _upsample_nn(coarse, N)


def _reconstruct_true_frac_maps(case_id: int, meta_local: dict) -> np.ndarray:
    """
    Rebuild true liquid-fraction maps from point data (trunk_xy, trunk_t, yf, case_ids).
    Assumes structured grid N x N at each save_time.
    """
    N        = int(meta_local["N"])
    times    = np.array(meta_local["save_times"], dtype=float)
    Nt       = len(times)

    # select all points belonging to this case
    mask_case = (case_ids == case_id)
    xy_case   = trunk_xy[mask_case]     # [Nt*N*N?, 2]
    t_case    = trunk_t[mask_case, 0]   # [Nt*N*N?]
    y_case    = yf[mask_case, 0]        # [Nt*N*N?]

    frac_maps = []
    for t in times:
        mask_t = np.isclose(t_case, t)
        xy_t   = xy_case[mask_t]           # [N*N, 2]
        y_t    = y_case[mask_t]            # [N*N]

        assert y_t.shape[0] == N * N, \
            f"Unexpected number of points for case {case_id}, time {t}: got {y_t.shape[0]}, expected {N*N}"

        # reshape to N x N; assumes same ordering as original mesh generation
        frac_maps.append(y_t.reshape(N, N).astype(np.float32))

    return np.stack(frac_maps, axis=0)    # [Nt, N, N]


@torch.no_grad()
def predict_case_maps(ckpt: Path, case_id: int, sel_times: np.ndarray):
    model, stats = load_trained(ckpt)
    meta_local = stats["meta"]
    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"])
    Ly  = float(meta_local["Ly"])

    # branch vecs for this case (Q, BC, T features)
    bQ_case  = branch_Q[case_id]    # [S_Q_all]
    bBC_case = branch_BC[case_id]   # [S_BC]
    bT_case  = branch_T[case_id]    # [S_T_all]
    bQn, bBCn, bTn = _norm_branch_for(stats, bQ_case, bBC_case, bT_case)

    bQ_t  = torch.from_numpy(bQn).to(device)   # [1, S_Q_all]
    bBC_t = torch.from_numpy(bBCn).to(device)  # [1, S_BC]
    bT_t  = torch.from_numpy(bTn).to(device)   # [1, S_T_all]

    # grid (N x N)
    xs = np.linspace(0.0, Lx, N, dtype=np.float32)
    ys = np.linspace(0.0, Ly, N, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    XY = np.stack([X, Y], axis=-1).reshape(-1, 2)  # [N*N, 2]
    XY_n = _norm_xy_for(stats, XY)                 # [N*N, 2]
    XY_t = torch.from_numpy(XY_n).to(device)

    outs = []
    for t in sel_times:
        tt = np.full((XY.shape[0], 1), float(t), dtype=np.float32)  # [N*N, 1]
        tt_n = _norm_t_for(stats, tt)                               # [N*N, 1]
        tt_t = torch.from_numpy(tt_n).to(device)

        # tile branch to match grid points
        B = XY_t.shape[0]
        bQ_tile  = bQ_t.repeat(B, 1)    # [B, S_Q_all]
        bBC_tile = bBC_t.repeat(B, 1)   # [B, S_BC]
        bT_tile  = bT_t.repeat(B, 1)    # [B, S_T_all]

        # model outputs normalized fraction → denormalize
        yhat_n = model(bQ_tile, bBC_tile, bT_tile, XY_t, tt_t)      # [B,1] normalized
        yhat_f = _denorm_y_for(stats, yhat_n).cpu().numpy()         # [B,1] fraction
        outs.append(yhat_f.reshape(N, N).astype(np.float32))
    return np.stack(outs, axis=0)  # [Tsel, N, N]


def _pick_times(times_all: np.ndarray, num_cols: int = 5, prefer: np.ndarray | None = None):
    """Pick ~evenly spaced time stamps (or nearest to 'prefer' if provided)."""
    times_all = np.array(times_all, dtype=float)
    if prefer is None:
        if len(times_all) <= num_cols:
            return times_all
        idx = np.linspace(0, len(times_all)-1, num_cols).round().astype(int)
        return times_all[idx]
    # map preferred to nearest in times_all
    out = []
    for t in prefer:
        out.append(times_all[np.argmin(np.abs(times_all - t))])
    # keep unique in order
    uniq = []
    for t in out:
        if len(uniq) == 0 or abs(uniq[-1] - t) > 1e-12:
            uniq.append(t)
    return np.array(uniq[:num_cols], dtype=float)


def plot_case_transient(ckpt_path: Path, case_id: int, num_cols: int = 5, prefer_times=None):
    meta_local = meta   # from DATA_FRAC
    N   = int(meta_local["N"])
    Lx  = float(meta_local["Lx"])
    Ly  = float(meta_local["Ly"])
    times_all = np.array(meta_local["save_times"], dtype=float)
    sel_times = _pick_times(times_all, num_cols=num_cols, prefer=prefer_times)

    # data: Q map, true fraction maps, predicted fraction maps
    Q_map        = _reconstruct_Q_map_from_branchQ(case_id, meta_local)      # [N,N]
    frac_true_all = _reconstruct_true_frac_maps(case_id, meta_local)         # [Nt,N,N]
    idx_true      = np.array([np.argmin(np.abs(times_all - t)) for t in sel_times], dtype=int)
    frac_true     = frac_true_all[idx_true]                                  # [Tsel,N,N]
    frac_pred     = predict_case_maps(ckpt_path, case_id, sel_times)         # [Tsel,N,N]

    # color scaling for fraction (0–1-ish)
    vmin = min(frac_true.min(), frac_pred.min())
    vmax = max(frac_true.max(), frac_pred.max())

    cols = len(sel_times)
    fig, axes = plt.subplots(nrows=3, ncols=cols, figsize=(3.2*cols, 9.0), constrained_layout=True)

    # Row 0: Heat source (repeat same Q for alignment)
    for c in range(cols):
        ax = axes[0, c]
        imQ = ax.imshow(Q_map.T, origin="lower", extent=[0, Lx, 0, Ly], cmap="RdBu_r", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("Heat source Q", fontsize=11)
        ax.set_title(f"t = {sel_times[c]:g}s", fontsize=11)
    fig.colorbar(imQ, ax=axes[0, :].ravel().tolist(), fraction=0.02, pad=0.02)

    # Row 1: True fraction
    for c in range(cols):
        ax = axes[1, c]
        imT = ax.imshow(frac_true[c].T, origin="lower", extent=[0, Lx, 0, Ly],
                        cmap="viridis", vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("True fraction", fontsize=11)
    fig.colorbar(imT, ax=axes[1, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("f_liq")

    # Row 2: Predicted fraction
    for c in range(cols):
        ax = axes[2, c]
        imP = ax.imshow(frac_pred[c].T, origin="lower", extent=[0, Lx, 0, Ly],
                        cmap="viridis", vmin=vmin, vmax=vmax, aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel("Pred fraction", fontsize=11)
    fig.colorbar(imP, ax=axes[2, :].ravel().tolist(), fraction=0.02, pad=0.02).set_label("f_liq")

    plt.show()

In [ ]:
# Pick a demo case and plot
ckpt_path = Path(r'runs/dataset_run_20251112-182232/deeponet_frac_model_5net.pt')

for i in range(len(test_ids)):
    demo_case = int(test_ids[i])  # 0-based indexing
    plot_case_transient(ckpt_path, demo_case, num_cols=7, prefer_times=save_times)
    print(demo_case + 1)          # print as 1-based for readability

In [ ]:
# ================================================================
# SENSITIVITY ANALYSIS (NO DATA REGEN):
#   - Use existing main dataset (e.g. 200 cases)
#   - For each combo: pick a subset of train_ids (e.g. 60, 80, 100 cases)
#   - Train DeepONet5 (liq fraction) from scratch
#   - Save model + liq-fraction transient plots under:
#       ~/Downloads/Sensitivity analysis/<run_tag>/
# ================================================================
# ------------------------------------------------------------
# 0) Base folder for sensitivity results
# ------------------------------------------------------------
SENS_BASE_DIR = Path.home() / "Downloads" / "Sensitivity analysis"
SENS_BASE_DIR.mkdir(parents=True, exist_ok=True)
print("Sensitivity results will be stored in:", SENS_BASE_DIR)

# ------------------------------------------------------------
# 1) Hyperparameter grid (KEPT SMALL)
# ------------------------------------------------------------

# Explicit number of training cases to use from main train_ids
# (will be clipped to len(train_ids) automatically)
CASE_COUNTS = [60, 80, 100, 120, 140, 160, 180]   # you can tweak these

# Learning rates
LR_LIST = [1e-5, 5e-4, 1e-4, 5e-3, 1e-3]

# Number of epochs
EPOCH_LIST = [50, 75, 100]

# Batch points (space-time points per virtual batch)
BATCH_LIST = [65536,65536//2, (65536//2)//2]

# Steps per epoch for each experiment (not full 1 epoch over all points — virtual)
SENS_STEPS_PER_EPOCH = 64

# Which test case to visualize for each run (index within test_ids)
DEMO_CASE_INDEX = 0

# Architecture configs: vary latent D and hidden layers
ARCH_CONFIGS = [
    {
        "name": "small",
        "D": 128,
        "Q_hidden":  (128, 128),
        "BC_hidden": None,          # will default to Q_hidden
        "T_hidden":  (128, 128),
        "xy_hidden": (128, 128, 128),
        "t_hidden":  (64, 64),
    },
    {
        "name": "base",
        "D": 256,
        "Q_hidden":  (256, 256, 256),
        "BC_hidden": None,
        "T_hidden":  (256, 256, 256),
        "xy_hidden": (256, 256, 256, 256),
        "t_hidden":  (128, 128),
    },
]

# ------------------------------------------------------------
# 2) Single experiment runner
# ------------------------------------------------------------
def run_sensitivity_experiment(run_tag: str,
                               train_ids_subset: np.ndarray,
                               lr: float,
                               epochs: int,
                               batch_points: int,
                               arch_cfg: dict):
    """
    For one hyperparameter + architecture combo:
      - Use a subset of train_ids (no data regeneration, no writing to dataset)
      - Train a new DeepONet5 model from scratch
      - Save:
          * checkpoint -> <run_dir>/model_frac_ckpt.pt
          * liq-fraction transient plot -> <run_dir>/frac_case_<id>_transient.png
      - Print simple-English summary.
    """
    # ------------ make folder for this run ------------
    run_dir = SENS_BASE_DIR / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)

    # ------------ describe architecture ------------
    D = arch_cfg["D"]
    Q_hid  = arch_cfg["Q_hidden"]
    BC_hid = arch_cfg["BC_hidden"] if arch_cfg["BC_hidden"] is not None else Q_hid
    T_hid  = arch_cfg["T_hidden"]
    xy_hid = arch_cfg["xy_hidden"]
    t_hid  = arch_cfg["t_hidden"]

    print("\n" + "="*80)
    print(f"Running experiment: {run_tag}")
    print(f"  → Training cases used        : {len(train_ids_subset)} / {len(train_ids)}")
    print(f"  → Learning rate              : {lr}")
    print(f"  → Epochs                     : {epochs}")
    print(f"  → Batch points               : {batch_points}")
    print(f"  → Steps per epoch            : {SENS_STEPS_PER_EPOCH}")
    print(f"  → Latent dimension D         : {D}")
    print(f"  → Q branch hidden layers     : {Q_hid}  (layers = {len(Q_hid)})")
    print(f"  → BC branch hidden layers    : {BC_hid} (layers = {len(BC_hid)})")
    print(f"  → T branch hidden layers     : {T_hid}  (layers = {len(T_hid)})")
    print(f"  → XY trunk hidden layers     : {xy_hid} (layers = {len(xy_hid)})")
    print(f"  → t trunk hidden layers      : {t_hid}  (layers = {len(t_hid)})")
    print("="*80 + "\n")

    # ------------ build datasets (liq fraction) ------------
    ds_tr = PooledPointDataset5(trunk_xy=trunk_xy, trunk_t=trunk_t, yf=yf, case_ids=case_ids,
                                branch_Q=branch_Q, branch_BC=branch_BC, branch_T=branch_T,
                                allowed_case_ids=train_ids_subset,
                                batch_points=batch_points)
    ds_va = PooledPointDataset5(trunk_xy=trunk_xy, trunk_t=trunk_t, yf=yf, case_ids=case_ids,
                                branch_Q=branch_Q, branch_BC=branch_BC, branch_T=branch_T,
                                allowed_case_ids=val_ids,
                                batch_points=batch_points)

    ds_tr.batch_points = batch_points
    ds_va.batch_points = batch_points

    # ------------ model for liq fraction ------------
    S_Q_all = branch_Q.shape[1]
    S_BC    = branch_BC.shape[1]
    S_T_all = branch_T.shape[1]

    model = DeepONet5(S_Q_all=S_Q_all,
                      S_BC=S_BC,
                      S_T_all=S_T_all,
                      D=D,
                      Q_hidden=Q_hid,
                      BC_hidden=BC_hid,
                      T_hidden=T_hid,
                      xy_hidden=xy_hid,
                      t_hidden=t_hid).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = torch.nn.MSELoss()

    ckpt_path = run_dir / "model_frac_ckpt.pt"
    best_val = math.inf

    # ------------ training loop ------------
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_acc = 0.0
        train_rmse_acc = 0.0

        for _ in range(SENS_STEPS_PER_EPOCH):
            bQ, bBC, bT, xy, tt, ytrue_n = ds_tr[0]
            bQ = bQ.to(device)
            bBC = bBC.to(device)
            bT = bT.to(device)
            xy = xy.to(device)
            tt = tt.to(device)
            ytrue_n = ytrue_n.to(device)

            opt.zero_grad(set_to_none=True)
            yhat_n = model(bQ, bBC, bT, xy, tt)
            loss = criterion(yhat_n, ytrue_n)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            with torch.no_grad():
                yhat_phys = denorm_y(yhat_n)
                y_phys    = denorm_y(ytrue_n)
                train_rmse_acc += torch.sqrt(((yhat_phys - y_phys)**2).mean()).item()
                train_loss_acc += loss.item()

        # ---- validation ----
        model.eval()
        vloss_acc = 0.0
        rmse_acc  = 0.0
        with torch.no_grad():
            for _ in range(VAL_SAMPLES):
                bQv, bBCv, bTv, xyv, ttv, yv_n = ds_va[0]
                bQv = bQv.to(device)
                bBCv = bBCv.to(device)
                bTv = bTv.to(device)
                xyv = xyv.to(device)
                ttv = ttv.to(device)
                yv_n = yv_n.to(device)

                yhatv_n = model(bQv, bBCv, bTv, xyv, ttv)
                vloss_acc += criterion(yhatv_n, yv_n).item()

                yhatv_phys = denorm_y(yhatv_n)
                yv_phys    = denorm_y(yv_n)
                rmse_acc += torch.sqrt(((yhatv_phys - yv_phys)**2).mean()).item()

        train_mse_n  = train_loss_acc / SENS_STEPS_PER_EPOCH
        train_rmse   = train_rmse_acc / SENS_STEPS_PER_EPOCH
        vloss        = vloss_acc / VAL_SAMPLES
        val_rmse     = rmse_acc / VAL_SAMPLES

        print(f"[{run_tag}] Epoch {epoch:03d} | "
              f"train MSE(n)={train_mse_n:.6e} | val MSE(n)={vloss:.6e} | "
              f"train RMSE={train_rmse:.4f} | val RMSE={val_rmse:.4f}")

        # track best
        if vloss < best_val:
            best_val = vloss
            torch.save({
                "model": model.state_dict(),
                "bQ_mean": bQ_mean,   "bQ_std":  bQ_std,
                "bBC_mean": bBC_mean, "bBC_std": bBC_std,
                "bT_mean": bT_mean,   "bT_std":  bT_std,
                "xy_min": xy_min,     "xy_max":  xy_max,
                "t_min": t_min,       "t_max":   t_max,
                "y_mean": y_mean,     "y_std":   y_std,
                "S_Q_all": S_Q_all,
                "S_BC":    S_BC,
                "S_T_all": S_T_all,
                "D":       D,
                "arch_name": arch_cfg["name"],
                "meta": meta,
                "run_tag": run_tag,
                "config": {
                    "train_cases": int(len(train_ids_subset)),
                    "lr": float(lr),
                    "epochs": int(epochs),
                    "batch_points": int(batch_points),
                    "steps_per_epoch": int(SENS_STEPS_PER_EPOCH),
                    "arch_name": arch_cfg["name"],
                    "D": int(D),
                    "Q_hidden": Q_hid,
                    "BC_hidden": BC_hid,
                    "T_hidden": T_hid,
                    "xy_hidden": xy_hid,
                    "t_hidden": t_hid,
                }
            }, ckpt_path)

    # ------------ simple-English summary for this combo ------------
    print("\nSummary for this hyperparameter setting (liq fraction):")
    print(f"  → Used {len(train_ids_subset)} training cases picked from the main dataset.")
    print(f"  → Learning rate {lr}, {epochs} epochs, batch size {batch_points}.")
    print(f"  → Architecture '{arch_cfg['name']}' with latent dimension D = {D}.")
    print(f"  → Best validation normalized MSE: {best_val:.3e}.")
    print("    In simple words: the smaller this MSE value is,")
    print("    the better the model matched the liquid-fraction maps "
          "for this particular choice of data amount and hyperparameters.\n")

    # ------------ save demo liq-fraction plot ------------
    if len(test_ids) > 0:
        demo_case_id = int(test_ids[DEMO_CASE_INDEX])

        # Uses your existing Step-6 function for liq fraction
        plot_case_transient(ckpt_path, demo_case_id, num_cols=5, prefer_times=save_times)
        fig_frac = plt.gcf()
        frac_fig_path = run_dir / f"frac_case_{demo_case_id}_transient.png"
        fig_frac.savefig(frac_fig_path, dpi=150, bbox_inches="tight")
        plt.close(fig_frac)
        print(f"Saved liq-fraction transient plot for case {demo_case_id} to:")
        print(f"  {frac_fig_path}")
    else:
        print("No test_ids available to plot an example case.")

    print("-"*80)
    print(f"Finished experiment: {run_tag}")
    print("-"*80 + "\n")

    return best_val


# ------------------------------------------------------------
# 3) Main sensitivity loop
# ------------------------------------------------------------
all_results = []

for n_target in CASE_COUNTS:
    n_cases = min(len(train_ids), n_target)
    if n_cases <= 0:
        continue

    # Deterministic choice: just take the first n_cases train_ids.
    # (If you want random subsets but reproducible, you can shuffle with a fixed seed.)
    subset_ids = train_ids[:n_cases]

    for arch_cfg in ARCH_CONFIGS:
        for lr in LR_LIST:
            for ep in EPOCH_LIST:
                for bs in BATCH_LIST:
                    run_tag = (f"cases{n_cases}_arch{arch_cfg['name']}"
                               f"_D{arch_cfg['D']}_lr{lr}_ep{ep}_bs{bs}")
                    best_val = run_sensitivity_experiment(run_tag, subset_ids, lr, ep, bs, arch_cfg)
                    all_results.append({
                        "run_tag": run_tag,
                        "train_cases": n_cases,
                        "arch_name": arch_cfg["name"],
                        "D": arch_cfg["D"],
                        "lr": lr,
                        "epochs": ep,
                        "batch_points": bs,
                        "best_val_mse": float(best_val),
                    })

print("\n================= OVERALL SENSITIVITY SUMMARY =================")
for r in all_results:
    print(
        f"{r['run_tag']}: "
        f"cases={r['train_cases']}, arch={r['arch_name']}, D={r['D']}, "
        f"lr={r['lr']}, epochs={r['epochs']}, batch={r['batch_points']} "
        f"→ best val MSE(n)={r['best_val_mse']:.3e}"
    )
print("===============================================================\n")
